# NB5 — Risk Profiles (Watershed L6 / ADM2 / ADM3) — EVT2-consistent YLT simulation (Option A)

**Inputs (from NB4):**
- EVT2 fit: `data/processed/impact_catalogue_catmodel/evt2/evt2_fit_popaffected_op.json`
- Reforecast event registries: `.../reforecast_library/event_registry_reforecast_library_*_with_full_rp.(parquet|csv)`
- Reforecast impacts by admin (monthly): `.../reforecast_library/impacts_by_admin_reforecast_library/year=YYYY/impacts_by_admin_YYYY_MM.(parquet|csv)`

**Key design:**
- EVT2 governs frequency/severity in impact space (fixed λ_total + tail/body via RP mapping).
- Reforecast library is a *pattern bank* (templates), not a frequency model.
- Sampling uses **RP-bin target mass weights** to ensure simulated severities respect EVT2.

**Notes:**
- Uses cached AOI/admin crosswalk when available (see `data/processed/Riskprofiles/cache`).
- EVT1 params are only used as a *fallback* to reconstruct basin AOI if no cached AOI exists.


In [ ]:
from pathlib import Path
import textwrap, datetime

REPO_ROOT = Path(r"C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL")
OUT_MD_DIR = REPO_ROOT / "data" / "processed" / "Riskprofiles"
OUT_MD_DIR.mkdir(parents=True, exist_ok=True)

context_md = f"""\
# NB5 CONTEXT — PhilFlood Risk Profiles (created {datetime.datetime.now().isoformat(timespec="seconds")})

## Output folder
- {OUT_MD_DIR}

## Hard contracts with NB4
- EVT2 fit must exist:
  - data/processed/impact_catalogue_catmodel/evt2/evt2_fit_popaffected_op.json
- Reforecast registry must exist (per year):
  - data/processed/impact_catalogue_catmodel/reforecast_library/event_registry_reforecast_library_<YEAR>_with_full_rp.(parquet|csv)
- Reforecast impacts by admin must exist (per month):
  - data/processed/impact_catalogue_catmodel/reforecast_library/impacts_by_admin_reforecast_library/year=<YEAR>/impacts_by_admin_<YEAR>_<MM>.(parquet|csv)

## Risk matrix denominator
- PopExposed500(unit) = WorldPop summed over pixels where (JRC RP500 depth > 0) within the unit & model AOI.

## External reference anchors
- JRC/CEMS GloFAS FloodHazard maps provide depths for return periods 10–500y.
- YLT/ELT are standard catastrophe modelling constructs.

## Key local paths (expected)
- Repo root: C:\\pipelines\\GLOFAS_ImpactFloodForecasting_PHL
- NB4 OUTPUT_DIR: data\\processed\\impact_catalogue_catmodel
- Risk profile outputs: data\\processed\\Riskprofiles
"""

readme_md = """\
# NB5 README — How to run

1) Run NB4 up to:
   - Section 6 (EVT2 saved to OUTPUT_DIR/evt2)
   - Section 7 (reforecast library + impacts_by_admin saved)
   - Section 7 “with_full_rp” mapping (writes event_registry_*_with_full_rp)

2) Open NB5 and run top-to-bottom.
   - It will self-discover years/months present in OUTPUT_DIR.
   - It will compute PopExposed500 denominators using WorldPop and the JRC RP500 wet mask.

3) Outputs:
   - Excel workbook in data/processed/Riskprofiles/
   - Cached intermediate tables (optional) in the same folder.

If something is missing:
- NB5 will raise a clear error indicating which NB4 artifact/path wasn’t found.
"""

(OUT_MD_DIR / "NB5_CONTEXT.md").write_text(context_md, encoding="utf-8")
(OUT_MD_DIR / "NB5_README.md").write_text(readme_md, encoding="utf-8")
print("Wrote:", OUT_MD_DIR / "NB5_CONTEXT.md")
print("Wrote:", OUT_MD_DIR / "NB5_README.md")

In [ ]:
from pathlib import Path
import os, json, re, glob
import numpy as np
import pandas as pd

# --- Repo layout (same as NB4) ---
REPO_ROOT = Path(r"C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL")
DATA_ROOT = REPO_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"

# --- NB4 output dir ---
NB4_OUTPUT_DIR = PROCESSED_ROOT / "impact_catalogue_catmodel"

# --- NB5 outputs ---
NB5_OUT_DIR = PROCESSED_ROOT / "Riskprofiles"
NB5_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Cache to make runs reproducible & faster (safe: does not change results)
CACHE_DIR = NB5_OUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
USE_CACHED_AOI = True
USE_CACHED_ADMIN_CROSSWALK = True

# --- PRIMARY DEPTH THRESHOLD ---
# Controls BOTH:
#   1. Which impact column is read from parquets  → PopAffected_{N}mm
#   2. Which EVT2 fit file is loaded              → evt2_fit_depth_{N}mm.json
#   3. Population denominator (exposed pop)       → JRC RP500 wet mask at ≥ this depth
# All three must use the same value so severity ratios stay in [0, 1].
# 0.2 m (200 mm) is the humanitarian standard for meaningful flood damage.
IMPACT_DEPTH_THR_M = 0.03

N_SIM_YEARS = 10_000
RNG_SEED = 42

# Curves reported up to 500y — starts at RP1 to cover the Moderate alert threshold
RP_REPORT = [1, 2, 5, 10, 20, 25, 50, 75, 100, 200, 500]

# Alert threshold RP targets for the RISK MATRIX EP table header rows
RP_TARGETS_MATRIX = [2, 5, 10]  # Moderate / High / Very High

# Risk-matrix severity bins as % of PopExposed at IMPACT_DEPTH_THR_M
SEV_PCT_BINS = [0.5, 1, 2, 5, 10]  # easy to adjust and sensitivity-test

# Risk-matrix likelihood bins — boundaries align with reference flood events
# Bin 0: RP≤2 (Moderate), Bin 1: RP 2–5 (High), Bin 2: RP 5–10 (Very High),
# Bin 3: RP 10–20 (below actionability), Bin 4: RP>20 (rare)
LIKELIHOOD_RP_BINS = [2, 5, 10, 20]  # defines: ≤2, 2–5, 5–10, 10–20, >20

# Absolute minimum impact floors — prevents misclassification for tiny populations
MIN_AAPA_PERSONS   = 100   # AAPA below this → force MONITOR regardless of ratio
MIN_IMPACT_PERSONS = 100   # OEP@RP25 below this → force Minor consequence bin

# Mock/smoke test toggle
RUN_MOCK_SMOKE_TEST = False

# --- Multi-threshold risk profile comparison (NB04 v0.5+ feature) ---
# When True: reads evt2_fit_manifest.json and runs YLT+OEP for ALL fitted depth thresholds,
# each using its own EVT2 fit, own impact column, and own per-event RP values.
# Plots a comparison chart and exports watershed_oep_multithr.json.
MULTI_THR_ENABLED = True

# --- NB5 v2 Excel UX toggles (do not affect results) ---
EXPORT_DASHBOARD = True
EXPORT_YLT_NONZERO = True
EXPORT_UNIT_COVERAGE_COLUMNS = True
EXPORT_QA_OEP_CHECK = True

ACTION_RP_CAP = 25  # used for dashboard headline and risk matrix narrative
TOP_N_MUNI_BAR = 10  # for dashboard bar chart

# --- Practitioner decision anchors ---
TRIGGER_RP = 2              # RP2 = Moderate alert threshold (aligns with reference events)
EXCLUDE_ZERO_IMPACT_UNITS = True  # remove "never affected" from UI

# --- NB6 event viewer intermediate file (optional — NB5 works without it) ---
# Run NB6 first to generate this file; NB5 will plot named events on the Risk Matrix if present
NB6_EVENTS_JSON = PROCESSED_ROOT / "event_viewer" / "events_for_risk_matrix.json"

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def safe_read_table(base: Path) -> pd.DataFrame:
    """Read parquet if available else CSV."""
    pq = base.with_suffix(".parquet")
    cs = base.with_suffix(".csv")
    if pq.exists():
        return pd.read_parquet(pq)
    if cs.exists():
        return pd.read_csv(cs)
    raise FileNotFoundError(f"Missing both: {pq} and {cs}")

def discover_run_config(processed_root: Path) -> dict:
    """Mimic NB4 fallback: pick latest run_config.json under calibration/evt_pot/*/*/."""
    candidates = list(processed_root.glob("calibration/evt_pot/*/*/run_config.json"))
    if not candidates:
        return {}
    latest = max(candidates, key=lambda p: p.stat().st_mtime)
    with open(latest, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    cfg["run_config_path"] = str(latest)
    return cfg

def discover_reforecast_registries(nb4_out: Path) -> list[Path]:
    """Find all event_registry_*_with_full_rp.(parquet|csv) under reforecast_library."""
    lib = nb4_out / "reforecast_library"
    pats = [
        str(lib / "event_registry_reforecast_library_*_with_full_rp.parquet"),
        str(lib / "event_registry_reforecast_library_*_with_full_rp.csv"),
    ]
    files = []
    for pat in pats:
        files.extend([Path(p) for p in glob.glob(pat)])
    by_stem = {}
    for f in files:
        by_stem.setdefault(f.stem, f)
    return sorted(by_stem.values())

def discover_impacts_by_admin(nb4_out: Path) -> list[Path]:
    """Find all impacts_by_admin_YYYY_MM.(parquet|csv) under impacts_by_admin_reforecast_library."""
    base = nb4_out / "reforecast_library" / "impacts_by_admin_reforecast_library"
    pats = [
        str(base / "year=*" / "impacts_by_admin_*.parquet"),
        str(base / "year=*" / "impacts_by_admin_*.csv"),
    ]
    files = []
    for pat in pats:
        files.extend([Path(p) for p in glob.glob(pat)])
    by_stem = {}
    for f in files:
        by_stem.setdefault(f.stem, f)
    return sorted(by_stem.values())

def enforce_monotone_nondec(x: np.ndarray) -> np.ndarray:
    """Monotone non-decreasing enforcement (cumulative max)."""
    y = np.asarray(x, float).copy()
    return np.maximum.accumulate(y)

def depth_thr_to_col(thr_m: float) -> str:
    """Convert depth threshold in metres to NB04 wide-format column name.

    Examples: 0.02 -> 'PopAffected_20mm', 0.2 -> 'PopAffected_200mm', 1.0 -> 'PopAffected_1000mm'
    Mirrors NB04's depth_thr_to_col() exactly so naming is consistent across notebooks.
    """
    mm = round(thr_m * 1000)
    return f"PopAffected_{mm}mm"

def depth_thr_to_file_tag(thr_m: float) -> str:
    """Convert depth threshold in metres to NB04 EVT2 file tag.

    Examples: 0.02 -> 'depth_20mm', 0.2 -> 'depth_200mm'
    Mirrors NB04's depth_thr_to_file_tag() so file discovery is consistent.
    """
    mm = round(thr_m * 1000)
    return f"depth_{mm}mm"

def compute_rp_from_evt2_spliced(
    impact_values: np.ndarray,
    fit: dict,
    rp_cap: float = 500.0,
) -> np.ndarray:
    """Compute return period for each impact value using the EVT2 spliced distribution.

    Uses:
    - Empirical body  (x ≤ u): Weibull plotting position on historical sample
    - GPD tail        (x > u): genpareto survival with xi=shape, sigma=scale

    Parameters
    ----------
    impact_values : array of watershed (or unit) impact values
    fit           : dict loaded from evt2_fit_depth_{N}mm.json
                    (keys: u, xi, sigma, lam_total, p_exc, hist)
    rp_cap        : hard upper cap on RP (avoids Inf in log space)

    Returns
    -------
    rp : array of same length as impact_values, values in [1, rp_cap]

    Note: consistent with NB04's impact_to_rp_spliced() and Cell 8 in this notebook.
    Sign convention: xi > 0 = heavy tail (matches scipy genpareto shape c directly).
    """
    u      = float(fit["u"])
    xi     = float(fit["xi"])
    sigma  = float(fit["sigma"])
    lam    = float(fit["lam_total"])
    p_exc  = float(fit["p_exc"])
    hist   = np.sort(np.asarray(fit.get("hist", []), dtype=float))
    n_h    = len(hist)

    impact_values = np.asarray(impact_values, dtype=float)
    rp_out = np.full(len(impact_values), 1.0, dtype=float)

    for i, x in enumerate(impact_values):
        if x <= 0:
            rp_out[i] = 1.0
            continue

        if n_h > 0 and x <= u:
            # Empirical body: Weibull plotting position
            j = int(np.searchsorted(hist, x, side="right"))
            p_emp = j / (n_h + 1)
            surv = max(1.0 - p_emp * (1.0 - p_exc), 1e-9)
        else:
            # GPD tail
            if abs(xi) > 1e-9:
                bracket = 1.0 + xi * (x - u) / sigma
                surv = p_exc * (max(bracket, 1e-12) ** (-1.0 / xi))
            else:
                surv = p_exc * np.exp(-(x - u) / sigma)
            surv = max(surv, 1e-9)

        rp_out[i] = min(1.0 / (lam * surv), rp_cap)

    return rp_out

def build_elt_for_threshold(
    thr_m: float,
    imp_files: list,
    event_set: set,
    muni_ids,
    col_event: str,
    col_muni: str,
) -> "pd.DataFrame | None":
    """Build an ELT (events × municipalities) matrix for a given depth threshold.

    Supports both NB04 v0.5+ wide-format (PopAffected_Nmm columns) and
    v0.4 long-format (depth_thr_m + affected_pop columns).
    Returns a DataFrame indexed by event_id with integer municipality ID columns,
    or None if no data found for this threshold.
    """
    col_thr = depth_thr_to_col(thr_m)
    chunks = []
    for f in imp_files:
        df = pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f)

        if col_thr in df.columns:
            # Wide-format (NB04 v0.5+)
            df = df[df[col_thr].astype(float) > 0]
            df = df[df[col_event].astype(str).isin(event_set)]
            if df.empty:
                continue
            chunk = df[[col_event, col_muni, col_thr]].rename(columns={col_thr: "_pop"})
        else:
            # Long-format fallback (NB04 v0.4 / current)
            col_depth_long = next(
                (c for c in ["depth_thr_m", "depth_threshold_m"] if c in df.columns), None
            )
            col_aff_long = next(
                (c for c in ["affected_pop", "pop_affected", "affected_p"] if c in df.columns), None
            )
            if col_depth_long is None or col_aff_long is None:
                continue
            mask = abs(df[col_depth_long].astype(float) - thr_m) < 1e-6
            df = df[mask & (df[col_aff_long].astype(float) > 0)]
            df = df[df[col_event].astype(str).isin(event_set)]
            if df.empty:
                continue
            chunk = df[[col_event, col_muni, col_aff_long]].rename(columns={col_aff_long: "_pop"})

        chunks.append(chunk)

    if not chunks:
        return None

    imp = pd.concat(chunks, ignore_index=True)
    imp[col_event] = imp[col_event].astype(str)
    imp[col_muni] = pd.to_numeric(imp[col_muni], errors="coerce")
    imp = imp[imp[col_muni].notna()].copy()
    imp[col_muni] = imp[col_muni].astype(int)
    imp = imp[imp[col_muni].isin(set(muni_ids))]

    if imp.empty:
        return None

    elt = (
        imp.pivot_table(index=col_event, columns=col_muni, values="_pop", aggfunc="sum", fill_value=0.0)
        .sort_index()
    )
    for mid in muni_ids:
        if mid not in elt.columns:
            elt[mid] = 0.0
    return elt[sorted(muni_ids)]

def rp_bin_weights_from_evt2_rp(
    rp_event: np.ndarray,
    lam_total: float,
    rp_bins: list[float],
    min_per_bin: int = 5,
) -> np.ndarray:
    """
    Option A weighting:
    - Bin events by their RP value
    - Target mass in bin derived from EVT2 exceedance implied by RP:
        exceed_prob(RP) ~ 1/(lam_total*RP)
      so mass([rmin,rmax]) = surv(rmin) - surv(rmax), with surv(r)=1/(lam*r) (clipped to [0,1]).
    - Each event in bin gets mass/n_bin
    - Auto-merge sparse bins from high→low if n_bin < min_per_bin.
    """
    rp_event = np.asarray(rp_event, float)
    ok = np.isfinite(rp_event) & (rp_event > 0)
    if ok.sum() == 0:
        raise ValueError("No finite positive RP values to weight.")
    rp = rp_event[ok]

    edges = sorted([float(b) for b in rp_bins])
    if edges[-1] != np.inf:
        edges = edges + [np.inf]

    def surv(r):
        if np.isinf(r):
            return 0.0
        return float(np.clip(1.0 / (lam_total * r), 0.0, 1.0))

    bin_ids = np.digitize(rp, edges, right=True) - 1
    n_bins = len(edges) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)
    counts = np.bincount(bin_ids, minlength=n_bins)

    merged_edges = edges.copy()
    while True:
        sparse = [b for b in range(n_bins-1, 0, -1) if counts[b] < min_per_bin]
        if not sparse:
            break
        b = sparse[0]
        bin_ids[bin_ids == b] = b-1
        bin_ids[bin_ids > b] -= 1
        del merged_edges[b]
        n_bins -= 1
        counts = np.bincount(bin_ids, minlength=n_bins)

    weights_ok = np.zeros_like(rp, dtype=float)
    for b in range(n_bins):
        rmin = merged_edges[b]
        rmax = merged_edges[b+1]
        mass = max(0.0, surv(rmin) - surv(rmax))
        nb = counts[b]
        if nb > 0 and mass > 0:
            weights_ok[bin_ids == b] = mass / nb

    wsum = weights_ok.sum()
    if wsum <= 0:
        raise ValueError("Computed weights sum to zero; check lam_total and RP bins.")
    weights_ok /= wsum

    weights = np.zeros_like(rp_event, dtype=float)
    weights[ok] = weights_ok
    s = weights.sum()
    weights = weights / s
    return weights

def simulate_ylt_shortform(
    impacts_evt_by_unit: np.ndarray,
    watershed_evt: np.ndarray,
    event_ids: np.ndarray,
    weights: np.ndarray,
    lam_total: float,
    n_years: int,
    seed: int,
):
    """
    Simulate short-form YLT:
    - N_y ~ Poisson(lam_total)
    - sample event indices with replacement using weights
    - compute annual sum & annual max per unit and watershed
    """
    rng = np.random.default_rng(seed)
    n_events, n_units = impacts_evt_by_unit.shape

    annual_sum = np.zeros((n_years, n_units), dtype=float)
    annual_max = np.zeros((n_years, n_units), dtype=float)

    ylt_rows = []
    for y in range(n_years):
        n = rng.poisson(lam_total)
        if n <= 0:
            ylt_rows.append((y+1, 0, 0.0, 0.0, None))
            continue
        idx = rng.choice(n_events, size=n, replace=True, p=weights)
        block = impacts_evt_by_unit[idx, :]
        s = block.sum(axis=0)
        m = block.max(axis=0)
        annual_sum[y, :] = s
        annual_max[y, :] = m

        w_block = watershed_evt[idx]
        imax = int(np.argmax(w_block))
        eid_max = str(event_ids[idx[imax]])
        ylt_rows.append((y+1, int(n), float(w_block.sum()), float(w_block[imax]), eid_max))

    ylt = pd.DataFrame(
        ylt_rows,
        columns=["year_idx", "n_events", "annual_sum_watershed", "annual_max_watershed", "event_id_of_max"],
    )
    return ylt, annual_sum, annual_max

def curves_from_annual(
    annual_sum: np.ndarray,
    annual_max: np.ndarray,
    rp_report: list[int],
):
    """Compute AEP (annual sum) and OEP (annual max) return levels for each unit."""
    rp = np.array(rp_report, dtype=float)
    q = np.clip(1.0 - 1.0 / rp, 0.0, 1.0)

    aapa = annual_sum.mean(axis=0)
    aep = np.quantile(annual_sum, q, axis=0).T
    oep = np.quantile(annual_max, q, axis=0).T
    aep = enforce_monotone_nondec(aep) if aep.ndim == 1 else np.vstack([enforce_monotone_nondec(aep[i]) for i in range(aep.shape[0])])
    oep = enforce_monotone_nondec(oep) if oep.ndim == 1 else np.vstack([enforce_monotone_nondec(oep[i]) for i in range(oep.shape[0])])
    return aapa, aep, oep

def invert_oep_curve_to_rp(impact_threshold: float, rp_grid: np.ndarray, oep_rl: np.ndarray) -> float:
    """Invert monotone OEP curve RL(RP) to get RP at which RL crosses impact_threshold."""
    rp = np.asarray(rp_grid, float)
    y = enforce_monotone_nondec(np.asarray(oep_rl, float))
    if impact_threshold <= y[0]:
        return float(rp[0])
    if impact_threshold >= y[-1]:
        return float(np.inf)
    j = int(np.searchsorted(y, impact_threshold, side="left"))
    j = max(1, min(j, len(y)-1))
    x0, x1 = np.log(rp[j-1]), np.log(rp[j])
    y0, y1 = y[j-1], y[j]
    if y1 == y0:
        return float(np.exp(x1))
    frac = (impact_threshold - y0) / (y1 - y0)
    return float(np.exp(x0 + frac * (x1 - x0)))

def likelihood_bin_from_rp(rp_value: float, rp_bins: list[float]) -> int:
    """Map RP value to 5-bin likelihood index."""
    if not np.isfinite(rp_value):
        return 4
    for i, b in enumerate(rp_bins):
        if rp_value <= b:
            return i
    return 4

In [ ]:
if RUN_MOCK_SMOKE_TEST:
    # Synthetic footprints
    rng = np.random.default_rng(0)
    n_events = 400
    n_muni = 30

    # Create a heavy-tailed RP distribution (roughly)
    rp_evt = np.exp(rng.normal(np.log(20), 1.2, size=n_events))
    rp_evt = np.clip(rp_evt, 1.5, 2000)

    lam_total = 2.0  # events/year
    rp_bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, np.inf]

    w = rp_bin_weights_from_evt2_rp(rp_evt, lam_total=lam_total, rp_bins=rp_bins, min_per_bin=5)
    assert np.isclose(w.sum(), 1.0), "Weights must sum to 1."

    # Muni impacts per event: sparse patterns, correlated with rarity
    severity = (1.0 / rp_evt)  # proxy
    base = rng.gamma(shape=2.0, scale=200.0, size=(n_events, n_muni))
    mask = rng.random((n_events, n_muni)) < 0.15
    impacts = base * mask * (1 + 50 * (severity[:, None]))  # rarer -> bigger
    watershed_evt = impacts.sum(axis=1)
    event_ids = np.array([f"evt_{i:05d}" for i in range(n_events)])

    # simulate
    ylt, ann_sum, ann_max = simulate_ylt_shortform(
        impacts_evt_by_unit=impacts,
        watershed_evt=watershed_evt,
        event_ids=event_ids,
        weights=w,
        lam_total=lam_total,
        n_years=5000,
        seed=1,
    )
    assert len(ylt) == 5000
    assert (ylt["annual_sum_watershed"] >= ylt["annual_max_watershed"]).all()

    # curves
    aapa, aep, oep = curves_from_annual(ann_sum, ann_max, rp_report=[1,2,5,10,20,25,50,100,200,500])
    assert np.all(np.diff(oep[0]) >= -1e-9), "OEP curve must be non-decreasing."
    assert np.all(np.diff(aep[0]) >= -1e-9), "AEP curve must be non-decreasing."

    print("✅ Mock smoke test passed.")

In [ ]:
# 1) Load run_config (dynamic, like NB4)
cfg = discover_run_config(PROCESSED_ROOT)
if not cfg:
    raise RuntimeError("No run_config.json found under processed/calibration/evt_pot/*/*/. Run NB1/NB4 first.")

print("Loaded run_config:", cfg.get("run_config_path"))
basin_id = cfg.get("basin_id", "unknown_basin")
run_tag  = cfg.get("run_tag", "unknown_runtag")

ADM3_GEOJSON = Path(cfg["adm3_geojson"])
EVT1_PARAMS_PARQUET = Path(cfg["evt_params_parquet"])

# 2) Load EVT2 fit for IMPACT_DEPTH_THR_M
# Try the depth-specific file first (NB04 v0.5+), fall back to legacy popaffected_op alias.
_fit_tag      = depth_thr_to_file_tag(IMPACT_DEPTH_THR_M)        # e.g. "depth_200mm"
_fit_specific = NB4_OUTPUT_DIR / "evt2" / f"evt2_fit_{_fit_tag}.json"
_fit_legacy   = NB4_OUTPUT_DIR / "evt2" / "evt2_fit_popaffected_op.json"

if _fit_specific.exists():
    evt2_fit_path = _fit_specific
elif _fit_legacy.exists():
    evt2_fit_path = _fit_legacy
    print(f"ℹ️  Depth-specific EVT2 fit not found ({_fit_specific.name}).")
    print(f"   Falling back to legacy popaffected_op.json.")
    print(f"   For full consistency, re-run NB4 with DEPTH_PRIMARY = {IMPACT_DEPTH_THR_M} m.")
else:
    raise FileNotFoundError(
        f"No EVT2 fit found for depth={IMPACT_DEPTH_THR_M}m. Tried:\n"
        f"  {_fit_specific}\n  {_fit_legacy}\n(Run NB4 Section 6.)"
    )

evt2 = json.loads(evt2_fit_path.read_text(encoding="utf-8"))
lam_total = float(evt2["lam_total"])

print(f"Basin: {basin_id}  |  Run tag: {run_tag}")
print(f"EVT2 fit loaded: {evt2_fit_path.name}  (depth={IMPACT_DEPTH_THR_M}m, lam_total={lam_total:.4f}/yr)")

# 3) Load EVT2 manifest (NB04 v0.5+) — used by multi-threshold comparison cell
evt2_manifest_path = NB4_OUTPUT_DIR / "evt2" / "evt2_fit_manifest.json"
if evt2_manifest_path.exists():
    _raw_manifest = json.loads(evt2_manifest_path.read_text(encoding="utf-8"))
    evt2_manifest = _raw_manifest.get("thresholds", _raw_manifest) if isinstance(_raw_manifest, dict) else _raw_manifest
    ok_count = sum(1 for e in evt2_manifest if e.get("status") == "OK")
    print(f"EVT2 manifest: {len(evt2_manifest)} thresholds total, {ok_count} OK")
else:
    evt2_manifest = []
    print("ℹ️  evt2_fit_manifest.json not found — multi-threshold comparison will be skipped.")

In [ ]:
import geopandas as gpd
import re
from pathlib import Path

# =========================
# 6) Build ELT (events x municipalities) + AOI selection (basin) — robust & cache-aware
# =========================

# --- Discover year registries ---
reg_files = discover_reforecast_registries(NB4_OUTPUT_DIR)
if not reg_files:
    raise RuntimeError(f"No event_registry_*_with_full_rp found under {NB4_OUTPUT_DIR}/reforecast_library/")

registries = []
for f in reg_files:
    df = pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f)
    registries.append(df)

rfc = pd.concat(registries, ignore_index=True)

# Dynamic impact column from IMPACT_DEPTH_THR_M — consistent with EVT2 fit and denominator
_primary_col = depth_thr_to_col(IMPACT_DEPTH_THR_M)  # e.g. "PopAffected_200mm"
if _primary_col in rfc.columns:
    _rfc_impact_col = _primary_col
elif "PopAffected_op" in rfc.columns:
    _rfc_impact_col = "PopAffected_op"
    print(f"ℹ️  Registry missing {_primary_col}; using PopAffected_op for event filter (backward compat).")
else:
    raise ValueError(f"Registry missing both {_primary_col} and PopAffected_op. Columns: {list(rfc.columns)}")

need_cols = {"event_id", _rfc_impact_col}
missing = need_cols - set(rfc.columns)
if missing:
    raise ValueError(f"Registry missing columns: {missing}")

rfc = rfc[(rfc[_rfc_impact_col] > 0)].copy()
# RP_impact_op_full may be absent for non-primary thresholds — we will recompute rp_evt below
# using compute_rp_from_evt2_spliced, so we only keep it if present (used as fallback check only)
event_set = set(rfc["event_id"].astype(str).tolist())

print(f"Registry events loaded: {len(rfc)} | impact col: {_rfc_impact_col}")
print(f"Years discovered: {sorted(rfc.get('year', pd.Series(dtype=int)).dropna().unique())[:10]}")

# --- Discover impacts_by_admin monthly files ---
imp_files = discover_impacts_by_admin(NB4_OUTPUT_DIR)
if not imp_files:
    raise RuntimeError("No impacts_by_admin_* found under impacts_by_admin_reforecast_library. Run NB4 Section 7.")

# --- Determine columns from first file ---
df0 = pd.read_parquet(imp_files[0]) if imp_files[0].suffix == ".parquet" else pd.read_csv(imp_files[0])

def guess_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

col_event = guess_col(df0, ["event_id", "EventID", "event"])
col_depth = guess_col(df0, ["depth_thr_m", "depth_threshold_m", "DEPTH_THR_M"])  # None in NB04 v0.5+ wide-format
col_muni  = guess_col(df0, ["adm3_id", "fid", "ADM3_ID", "admin3_id", "admin_id"])
# Dynamic: prefer the exact IMPACT_DEPTH_THR_M column, fall back to PopAffected_op alias
col_aff   = guess_col(df0, [depth_thr_to_col(IMPACT_DEPTH_THR_M), "PopAffected_op",
                             "affected_pop", "pop_affected", "people_affected"])

WIDE_FORMAT = col_depth is None
if WIDE_FORMAT:
    print(f"ℹ️  Wide-format parquets (NB04 v0.5+): using col_aff={col_aff}, no depth row filter.")
else:
    print(f"ℹ️  Long-format parquets: depth filter col={col_depth}, value={IMPACT_DEPTH_THR_M}")

for name, col in [("event_id", col_event), ("muni_id", col_muni), ("affected_pop", col_aff)]:
    if col is None:
        raise ValueError(f"Could not find required column for {name}. Columns: {list(df0.columns)}")

# --- Load impacts ---
imp_keep = []
for f in imp_files:
    df = pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f)
    if not WIDE_FORMAT:
        df = df[df[col_depth].astype(float) == float(IMPACT_DEPTH_THR_M)]
    df = df[df[col_aff].astype(float) > 0]
    df = df[df[col_event].astype(str).isin(event_set)]
    imp_keep.append(df[[col_event, col_muni, col_aff]].copy())

imp = pd.concat(imp_keep, ignore_index=True)
imp[col_event] = imp[col_event].astype(str)

# --- Load ADM3 geojson ---
admin_gdf = gpd.read_file(ADM3_GEOJSON)
admin_gdf.columns = admin_gdf.columns.str.strip()
admin_gdf = admin_gdf.to_crs("EPSG:4326").reset_index(drop=True)

ADMIN3_ID_FIELD = next((c for c in admin_gdf.columns if c.lower() in ("adm3_id","adm3_pcode") or "pcode" in c.lower()), None)
ADMIN3_NAME_FIELD = next((c for c in admin_gdf.columns if c.lower() in ("adm3_name","adm3_en","adm3_name")), None)
ADM2_NAME_COL = "adm2_name"

if ADMIN3_ID_FIELD is None:
    raise ValueError(f"Could not find admin ID field in columns: {list(admin_gdf.columns)}")
if ADMIN3_NAME_FIELD is None:
    raise ValueError(f"Could not find admin NAME field in columns: {list(admin_gdf.columns)}")
if ADM2_NAME_COL not in admin_gdf.columns:
    raise ValueError(f"ADM3 geojson missing '{ADM2_NAME_COL}'. Columns: {list(admin_gdf.columns)}")

admin_gdf["original_id"] = admin_gdf[ADMIN3_ID_FIELD].astype(str)
ADM3_NAME_COL = ADMIN3_NAME_FIELD

crosswalk_cache = CACHE_DIR / f"adm3_crosswalk_{basin_id}_{run_tag}.parquet"
if USE_CACHED_ADMIN_CROSSWALK and crosswalk_cache.exists():
    xw = pd.read_parquet(crosswalk_cache)
    if not {"original_id","adm3_id"}.issubset(xw.columns):
        raise ValueError(f"Invalid crosswalk cache schema: {crosswalk_cache}")
    mapping = dict(zip(xw["original_id"].astype(str), xw["adm3_id"].astype(int)))
    admin_gdf["adm3_id"] = admin_gdf["original_id"].map(mapping)
    if admin_gdf["adm3_id"].isna().any():
        missing = admin_gdf.loc[admin_gdf["adm3_id"].isna(), "original_id"].head(10).tolist()
        raise RuntimeError(f"Cached ADM3 crosswalk missing some original_id values (first 10): {missing}")
    admin_gdf["adm3_id"] = admin_gdf["adm3_id"].astype(int)
    print(f"✓ Loaded cached ADM3 crosswalk: {crosswalk_cache}")
else:
    admin_gdf["adm3_id"] = pd.factorize(admin_gdf[ADMIN3_ID_FIELD])[0] + 1
    admin_gdf["adm3_id"] = admin_gdf["adm3_id"].astype(int)
    xw = admin_gdf[["original_id","adm3_id", ADM3_NAME_COL, ADM2_NAME_COL]].drop_duplicates("original_id").copy()
    xw.to_parquet(crosswalk_cache, index=False)
    print(f"✓ Saved ADM3 crosswalk cache: {crosswalk_cache}")

ADM3_ID_COL = "adm3_id"

imp[col_muni] = pd.to_numeric(imp[col_muni], errors="coerce")
imp = imp[imp[col_muni].notna()].copy()
imp[col_muni] = imp[col_muni].astype(int)

overlap = len(set(imp[col_muni].unique()).intersection(set(admin_gdf[ADM3_ID_COL].unique())))
print(f"Overlap impacts IDs vs admin numeric IDs: {overlap} / {imp[col_muni].nunique()}")
if overlap == 0:
    raise RuntimeError(
        "No overlap between impacts muni IDs and admin numeric IDs. "
        "This likely means NB4 used a different admin file or different factorization order."
    )

# --- Basin AOI boundary ---
aoi_cache = CACHE_DIR / f"basin_boundary_{basin_id}_{run_tag}.geojson"
if USE_CACHED_AOI and aoi_cache.exists():
    aoi_gdf = gpd.read_file(aoi_cache).to_crs("EPSG:4326")
    basin_boundary = aoi_gdf.unary_union
    print(f"✓ Loaded cached basin boundary: {aoi_cache}")
else:
    def parse_cell_coords(cell_id: str):
        m = re.search(r"lat_([-\d.]+)__lon_([-\d.]+)", str(cell_id))
        if not m:
            return (None, None)
        return float(m.group(1).rstrip(".")), float(m.group(2).rstrip("."))

    evt_params = pd.read_parquet(EVT1_PARAMS_PARQUET).rename(columns={
        "virtual_gauge_id": "cell_id", "threshold_m3s": "u",
        "lambda_events_per_year": "lam", "gpd_xi": "xi", "gpd_sigma": "sigma",
    })
    if "cell_id" not in evt_params.columns:
        raise ValueError(f"EVT1 params parquet missing 'cell_id'. Columns: {list(evt_params.columns)}")

    if ("lat" not in evt_params.columns) or ("lon" not in evt_params.columns):
        coords = evt_params["cell_id"].astype(str).apply(parse_cell_coords)
        evt_params["lat"] = coords.apply(lambda t: t[0])
        evt_params["lon"] = coords.apply(lambda t: t[1])

    bad = evt_params["lat"].isna() | evt_params["lon"].isna()
    if bad.any():
        raise ValueError(f"Could not derive lat/lon for some EVT1 gauges: {evt_params.loc[bad,'cell_id'].head(10).tolist()}")

    evt_pts = gpd.GeoDataFrame(
        evt_params.copy(),
        geometry=gpd.points_from_xy(evt_params["lon"], evt_params["lat"]),
        crs="EPSG:4326",
    )

    HYBAS_DIR = RAW_ROOT / "vectors" / "hydrobasins" / "australasia" / "hybas_au_lev01-12_v1c"
    if not HYBAS_DIR.exists():
        raise FileNotFoundError(f"HydroBASINS folder not found: {HYBAS_DIR}")
    hybas_candidates = list(HYBAS_DIR.glob("*lev06*.*"))
    if not hybas_candidates:
        raise FileNotFoundError(f"No lev06 HydroBASINS file found in: {HYBAS_DIR}")

    minx, miny, maxx, maxy = evt_pts.total_bounds
    pad = 0.5
    hybas = gpd.read_file(hybas_candidates[0], bbox=(minx-pad, miny-pad, maxx+pad, maxy+pad)).to_crs("EPSG:4326")
    join = gpd.sjoin(hybas, evt_pts, predicate="intersects", how="inner")
    if join.empty:
        raise RuntimeError("No HydroBASINS L6 polygons intersect EVT points; check CRS or HydroBASINS region file.")

    basin_boundary = join.geometry.unary_union
    gpd.GeoDataFrame({"basin_id":[basin_id], "run_tag":[run_tag]}, geometry=[basin_boundary], crs="EPSG:4326").to_file(aoi_cache, driver="GeoJSON")
    print(f"✓ Saved cached basin boundary: {aoi_cache}")

# --- Select basin municipalities ---
muni_in_basin = admin_gdf[admin_gdf.intersects(basin_boundary)].copy()
muni_ids = muni_in_basin[ADM3_ID_COL].astype(int).unique()
print(f"Municipalities in basin: {len(muni_ids)}")

imp = imp[imp[col_muni].isin(set(muni_ids))].copy()

# --- Pivot to ELT matrix ---
elt_muni = (
    imp.pivot_table(index=col_event, columns=col_muni, values=col_aff, aggfunc="sum", fill_value=0.0)
    .sort_index()
)
for mid in muni_ids:
    if mid not in elt_muni.columns:
        elt_muni[mid] = 0.0
elt_muni = elt_muni[sorted(muni_ids)]

# Watershed total per event
watershed_evt = elt_muni.sum(axis=1).astype(float).values

# --- Compute per-event RP using the loaded EVT2 fit ---
# This is fully dynamic: always uses evt2 (loaded from the file matching IMPACT_DEPTH_THR_M).
# Replaces registry's RP_impact_op_full (which was for NB04's primary threshold, not necessarily ours).
rp_evt = compute_rp_from_evt2_spliced(watershed_evt, evt2)
print(f"Per-event RP recomputed from {evt2_fit_path.name}  "
      f"(median={np.nanmedian(rp_evt):.1f}y, max={np.nanmax(rp_evt):.0f}y)")

# Also keep evt_tbl aligned for downstream use (primarily event index alignment)
evt_tbl = rfc.set_index("event_id")[[_rfc_impact_col]].copy()
evt_tbl.index = evt_tbl.index.astype(str)
evt_tbl = evt_tbl.loc[elt_muni.index]

# Province rollup
fid_to_adm2 = muni_in_basin.set_index(ADM3_ID_COL)[ADM2_NAME_COL].to_dict()
prov_names = sorted(set(fid_to_adm2.values()))
fid_list = list(elt_muni.columns.astype(int))

prov_mat = np.zeros((len(fid_list), len(prov_names)), dtype=float)
for j, fid in enumerate(fid_list):
    p = fid_to_adm2[int(fid)]
    k = prov_names.index(p)
    prov_mat[j, k] = 1.0

elt_prov = elt_muni.values @ prov_mat

admin_id_to_name = dict(zip(admin_gdf[ADM3_ID_COL].astype(int), admin_gdf[ADM3_NAME_COL].astype(str)))

# UI exclusion rule
max_evt_muni = elt_muni.max(axis=0).astype(float)
max_evt_prov = pd.Series(elt_prov.max(axis=0), index=prov_names).astype(float)

keep_muni = (max_evt_muni > 0)
keep_prov = (max_evt_prov > 0)

excluded_muni_ids   = max_evt_muni.index[~keep_muni].astype(int).tolist()
excluded_prov_names = max_evt_prov.index[~keep_prov].tolist()

print(f"ELT shapes: muni {elt_muni.shape} | prov {elt_prov.shape} | watershed {watershed_evt.shape}")
print(f"Excluded (max impact=0): munis {len(excluded_muni_ids)} | prov {len(excluded_prov_names)}")

In [ ]:
# TEMP DIAGNOSTIC: Why can DEPTH_THR_M=0.01 produce lower AAPA than 0.2?
# This checks float-matching effects and data coverage by threshold.

import numpy as np
import pandas as pd

thr_a = 0.2
thr_b = 0.01
tol = 1e-6

rows = []
for f in imp_files:
    d = pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f)

    depth = pd.to_numeric(d[col_depth], errors="coerce")
    aff = pd.to_numeric(d[col_aff], errors="coerce")
    evt = d[col_event].astype(str)

    base = aff.gt(0) & evt.isin(event_set) & depth.notna()

    exact_a = base & depth.eq(thr_a)
    exact_b = base & depth.eq(thr_b)
    near_a = base & np.isclose(depth, thr_a, atol=tol, rtol=0.0)
    near_b = base & np.isclose(depth, thr_b, atol=tol, rtol=0.0)

    rows.append({
        "file": f.name,
        "rows_total": int(len(d)),
        "rows_base": int(base.sum()),
        "exact_0.2_rows": int(exact_a.sum()),
        "exact_0.01_rows": int(exact_b.sum()),
        "near_0.2_rows": int(near_a.sum()),
        "near_0.01_rows": int(near_b.sum()),
        "exact_0.2_events": int(evt[exact_a].nunique()),
        "exact_0.01_events": int(evt[exact_b].nunique()),
        "near_0.2_events": int(evt[near_a].nunique()),
        "near_0.01_events": int(evt[near_b].nunique()),
        "exact_0.2_pop": float(aff[exact_a].sum()),
        "exact_0.01_pop": float(aff[exact_b].sum()),
        "near_0.2_pop": float(aff[near_a].sum()),
        "near_0.01_pop": float(aff[near_b].sum()),
    })

_diag = pd.DataFrame(rows)

# Aggregate totals
num_cols = [c for c in _diag.columns if c != "file"]
tot = _diag[num_cols].sum(numeric_only=True).to_dict()

summary = pd.DataFrame([
    {
        "metric": "rows",
        "exact_0.2": int(tot.get("exact_0.2_rows", 0)),
        "exact_0.01": int(tot.get("exact_0.01_rows", 0)),
        "near_0.2": int(tot.get("near_0.2_rows", 0)),
        "near_0.01": int(tot.get("near_0.01_rows", 0)),
    },
    {
        "metric": "events (sum across files)",
        "exact_0.2": int(tot.get("exact_0.2_events", 0)),
        "exact_0.01": int(tot.get("exact_0.01_events", 0)),
        "near_0.2": int(tot.get("near_0.2_events", 0)),
        "near_0.01": int(tot.get("near_0.01_events", 0)),
    },
    {
        "metric": "affected_pop (sum across files)",
        "exact_0.2": float(tot.get("exact_0.2_pop", 0.0)),
        "exact_0.01": float(tot.get("exact_0.01_pop", 0.0)),
        "near_0.2": float(tot.get("near_0.2_pop", 0.0)),
        "near_0.01": float(tot.get("near_0.01_pop", 0.0)),
    },
])

print("Diagnostic tolerance:", tol)
print("\nPer-file diagnostics (first 12 rows):")
display(_diag.head(12))
print("\nAggregate comparison:")
display(summary)

# Quick precision check: values very close to 0.01 but not exactly equal
_depth_samples = []
for f in imp_files[:8]:
    d = pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f)
    vals = pd.to_numeric(d[col_depth], errors="coerce")
    near = vals[np.isclose(vals, thr_b, atol=tol, rtol=0.0)]
    if len(near) > 0:
        _depth_samples.extend([float(v) for v in near.iloc[:5]])

if _depth_samples:
    print("\nSample depth_thr_m values near 0.01:", [round(v, 12) for v in _depth_samples[:15]])
else:
    print("\nNo values near 0.01 found in sampled files.")

In [ ]:
# ── Optional: recalculate RP_impact_op_full from updated EVT2 GPD parameters ──
# Set True when you've edited evt2_fit_popaffected_op.json without re-running NB04.
RECALCULATE_RP_FROM_EVT2 = True

if RECALCULATE_RP_FROM_EVT2:
    from scipy import stats as _stats

    # Reload the (possibly updated) file fresh from disk
    _e = json.loads(evt2_fit_path.read_text(encoding="utf-8"))
    _u       = float(_e["u"])
    _xi      = float(_e["xi"])
    _sigma   = float(_e["sigma"])
    _lam     = float(_e["lam_total"])
    _p_exc   = float(_e["p_exc"])
    _hist    = np.array(_e["hist"], dtype=float)
    _RP_CAP  = 500.0

    # ── Body: empirical Weibull plotting position ──
    _hist_body = np.sort(_hist[_hist <= _u])
    _n_body    = len(_hist_body)
    _n_total   = len(_hist)
    _lam_body  = _lam * (1.0 - _p_exc)          # arrival rate of body events

    def _rp_body(x_arr: np.ndarray) -> np.ndarray:
        out = np.full_like(x_arr, np.nan, dtype=float)
        ok  = np.isfinite(x_arr) & (x_arr > 0) & (x_arr <= _u)
        if ok.any() and _n_body >= 2:
            r   = np.searchsorted(_hist_body, x_arr[ok], side="right")   # count ≤ x
            exc = (_n_body - r) / (_n_body + 1.0)                        # Weibull exceedance in body
            exc = np.clip(exc, 1e-9, 1.0)
            out[ok] = np.clip(1.0 / (_lam_body * exc), 1.0, _RP_CAP)
        return out

    # ── Tail: GPD survival ──
    def _rp_tail(x_arr: np.ndarray) -> np.ndarray:
        out = np.full_like(x_arr, np.nan, dtype=float)
        ok  = np.isfinite(x_arr) & (x_arr > _u)
        if ok.any():
            y = (x_arr[ok] - _u) / _sigma
            if abs(_xi) < 1e-8:
                surv = np.exp(-y)
            else:
                bracket = 1.0 + _xi * y
                surv    = np.where(bracket > 0, bracket ** (-1.0 / _xi), 0.0)
            rate    = _lam * _p_exc * surv
            out[ok] = np.where(rate > 0, np.clip(1.0 / rate, 1.0, _RP_CAP), _RP_CAP)
        return out

    # ── Apply to all events in the loaded registry ──
    _x        = rfc["PopAffected_op"].astype(float).values
    _rp_b     = _rp_body(_x)
    _rp_t     = _rp_tail(_x)
    _rp_full  = _rp_b.copy()
    _mask_t   = np.isfinite(_rp_t)
    _rp_full[_mask_t] = _rp_t[_mask_t]

    rfc = rfc.copy()
    rfc["RP_impact_op_full"] = _rp_full

    # Keep lam_total in sync with the updated json
    lam_total = _lam

    # Re-derive rp_evt (used by weighting & simulation cells)
    evt_tbl   = rfc.set_index("event_id")[["RP_impact_op_full", "PopAffected_op"]].copy()
    evt_tbl.index = evt_tbl.index.astype(str)
    evt_tbl   = evt_tbl.loc[elt_muni.index]
    rp_evt    = evt_tbl["RP_impact_op_full"].astype(float).values

    print(f"RP recalculated from updated EVT2 params (json: {evt2_fit_path.name})")
    print(f"  u={_u:.0f}, xi={_xi:.4f}, sigma={_sigma:.0f}, lam_total={_lam:.3f}/y, p_exc={_p_exc:.3f}")
    print(f"  Tail events (GPD applied): {_mask_t.sum()} / {len(rfc)}")
    print(f"  RP range: [{np.nanmin(_rp_full):.1f}, {np.nanmax(_rp_full):.1f}]")
    print("  ✅ rp_evt overridden — re-run weighting + simulation cells.")
else:
    print("Using original RP_impact_op_full from parquet/CSV files.")

In [ ]:
# --- DIAGNOSTIC: Compare NB5-generated numeric IDs vs Impact table IDs, and validate name match ---

# 0) Confirm assumptions
print("col_muni used in impacts:", col_muni)
print("Unique muni IDs in impacts (within depth thr):", imp[col_muni].nunique())

# 1) Prepare NB5 crosswalk: numeric_id -> names
nb5_xwalk = (
    admin_gdf[[ADM3_ID_COL, ADM3_NAME_COL, ADM2_NAME_COL, "original_id"]]
    .rename(columns={ADM3_ID_COL: "adm3_id_num"})
    .drop_duplicates("adm3_id_num")
    .copy()
)

# 2) Prepare impacts ID list
imp_ids = pd.to_numeric(imp[col_muni], errors="coerce").dropna().astype(int)
imp_id_df = pd.DataFrame({"adm3_id_num": sorted(set(imp_ids.unique().tolist()))})

# 3) Join impacts IDs to NB5 mapping
joined = imp_id_df.merge(nb5_xwalk, on="adm3_id_num", how="left")

missing = joined["adm3_name"].isna().sum()
print("Impacts IDs that do NOT exist in NB5 admin mapping:", missing, "/", len(joined))

if missing > 0:
    print("Example missing IDs:", joined.loc[joined["adm3_name"].isna(), "adm3_id_num"].head(20).tolist())

# 4) If impacts has names, do strict match check
impact_name_cols = [c for c in imp.columns if c.lower() in ("adm3_name","adm2_name","province","municipality")]
print("Impact table candidate name columns:", impact_name_cols)

if "adm3_name" in imp.columns:
    # build impacts name map (most frequent name per id)
    imp_name_map = (
        imp.assign(adm3_id_num=pd.to_numeric(imp[col_muni], errors="coerce"))
           .dropna(subset=["adm3_id_num"])
           .assign(adm3_id_num=lambda d: d["adm3_id_num"].astype(int))
           .groupby("adm3_id_num")["adm3_name"]
           .agg(lambda s: s.value_counts().index[0])
           .reset_index()
    )
    joined2 = joined.merge(imp_name_map, on="adm3_id_num", how="left", suffixes=("_nb5", "_imp"))

    # After your joined2 is created (or recreate it cleanly):

    imp_name_map = (
        imp.assign(adm3_id_num=pd.to_numeric(imp[col_muni], errors="coerce"))
        .dropna(subset=["adm3_id_num"])
        .assign(adm3_id_num=lambda d: d["adm3_id_num"].astype(int))
        .groupby("adm3_id_num")["adm3_name"]
        .agg(lambda s: s.value_counts().index[0])
        .reset_index()
        .rename(columns={"adm3_name": "adm3_name_imp"})
    )

    # Build NB5 xwalk with explicit column name to avoid suffix confusion
    nb5_xwalk = (
        admin_gdf[[ADM3_ID_COL, ADM3_NAME_COL, ADM2_NAME_COL, "original_id"]]
        .rename(columns={ADM3_ID_COL: "adm3_id_num", ADM3_NAME_COL: "adm3_name_nb5", ADM2_NAME_COL: "adm2_name_nb5"})
        .drop_duplicates("adm3_id_num")
    )

    joined2 = (
        pd.DataFrame({"adm3_id_num": sorted(set(pd.to_numeric(imp[col_muni], errors="coerce").dropna().astype(int)))})
        .merge(nb5_xwalk, on="adm3_id_num", how="left")
        .merge(imp_name_map, on="adm3_id_num", how="left")
    )

    print("Columns in joined2:", list(joined2.columns))

    joined2["name_match"] = joined2["adm3_name_nb5"].fillna("") == joined2["adm3_name_imp"].fillna("")
    mismatch = joined2[~joined2["name_match"] & joined2["adm3_name_imp"].notna()]

    print("Name mismatches:", len(mismatch), "/", len(joined2))
    display(mismatch.head(50))
    display(joined2.sample(min(15, len(joined2)), random_state=0))

    mismatch = joined2[~joined2["name_match"] & joined2["adm3_name_imp"].notna()]
    print("Name mismatches (if any):", len(mismatch))
    display(mismatch.head(30))
else:
    print("No adm3_name in impacts file -> cannot do strict name match. Proceeding with coverage + spot-check export.")

# 5) Spot-check: show a few IDs with names so you can visually verify
display(joined.sample(min(15, len(joined)), random_state=0))

# 6) Optional: export a crosswalk for human audit (small)
xwalk_out = NB5_OUT_DIR / f"adm3_id_crosswalk_{basin_id}_{run_tag}.csv"
joined.to_csv(xwalk_out, index=False)
print("Wrote crosswalk:", xwalk_out)

In [ ]:
# Weight bins (transparent, stable)
RP_BINS_FOR_WEIGHTS = [1, 2, 5, 10, 20, 50, 100, 200, 500, np.inf]

w = rp_bin_weights_from_evt2_rp(
    rp_event=rp_evt,
    lam_total=lam_total,
    rp_bins=RP_BINS_FOR_WEIGHTS,
    min_per_bin=5,  # auto-merge sparse bins
)

event_ids = elt_muni.index.astype(str).values

# Build full unit matrix: [munis | provinces | watershed]
unit_names = []
muni_names = [muni_in_basin.set_index(ADM3_ID_COL).loc[int(fid), ADM3_NAME_COL] for fid in fid_list]
unit_names.extend([f"ADM3::{n}" for n in muni_names])
unit_names.extend([f"ADM2::{p}" for p in prov_names])
unit_names.append("WATERSHED::TOTAL")

impacts_evt_by_unit = np.hstack([
    elt_muni.values,            # events x munis
    elt_prov,                   # events x provinces
    watershed_evt[:, None],     # events x 1
])

# simulate
ylt_df, annual_sum, annual_max = simulate_ylt_shortform(
    impacts_evt_by_unit=impacts_evt_by_unit,
    watershed_evt=watershed_evt,
    event_ids=event_ids,
    weights=w,
    lam_total=lam_total,
    n_years=N_SIM_YEARS,
    seed=RNG_SEED,
)

print("YLT rows:", len(ylt_df))
display(ylt_df.head())

In [ ]:
aapa, aep_rl, oep_rl = curves_from_annual(annual_sum, annual_max, rp_report=RP_REPORT)

# Package outputs into tables
rp_cols = [f"RP{int(r)}" for r in RP_REPORT]

tbl_aapa = pd.DataFrame({"unit": unit_names, "AAPA": aapa})
tbl_aep  = pd.DataFrame(aep_rl, columns=rp_cols)
tbl_oep  = pd.DataFrame(oep_rl, columns=rp_cols)
tbl_aep.insert(0, "unit", unit_names)
tbl_oep.insert(0, "unit", unit_names)

display(tbl_aapa.head())
display(tbl_oep.head())

In [ ]:
# =============================================================================
# Multi-threshold OEP comparison (NB04 v0.5+ wide-format)
# Controlled by MULTI_THR_ENABLED in CONFIG cell.
#
# Fully dynamic per threshold:
#   - Impact column  → depth_thr_to_col(thr_m)        e.g. PopAffected_200mm
#   - EVT2 fit       → evt2_fit_depth_{N}mm.json       own u, xi, sigma, lam_total
#   - Per-event RP   → compute_rp_from_evt2_spliced()  uses that threshold's own fit
#   - RP weights     → rp_bin_weights_from_evt2_rp()   uses that threshold's lam_total
# The only shared input is the event set (discharge-detected events from NB04).
#
# Outputs:
#   - Overlay chart of watershed OEP curves (IMPACT_DEPTH_THR_M highlighted)
#   - data/processed/Riskprofiles/watershed_oep_multithr.json
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.cm as cm

multi_thr_oep = {}   # thr_m -> {"oep_watershed": [...], "lam_total": float, "n_events": int}

if not MULTI_THR_ENABLED:
    print("ℹ️  MULTI_THR_ENABLED = False — multi-threshold comparison skipped.")
elif not evt2_manifest:
    print("ℹ️  evt2_manifest is empty — multi-threshold comparison skipped (run NB4 first).")
else:
    ok_entries = [e for e in evt2_manifest if e.get("status") == "OK"]
    print(f"Running multi-threshold comparison: {len(ok_entries)} OK thresholds")

    for entry in ok_entries:
        thr_m    = float(entry["depth_m"])
        col_thr  = depth_thr_to_col(thr_m)
        fit_file = NB4_OUTPUT_DIR / "evt2" / entry["file"]

        if not fit_file.exists():
            print(f"  ⚠️  Missing fit file for {thr_m}m ({fit_file.name}) — skipped")
            continue

        fit_thr  = json.loads(fit_file.read_text(encoding="utf-8"))
        lam_thr  = float(fit_thr["lam_total"])

        # Build ELT using this threshold's own impact column
        elt_thr = build_elt_for_threshold(
            thr_m=thr_m, imp_files=imp_files, event_set=event_set,
            muni_ids=muni_ids, col_event=col_event, col_muni=col_muni,
        )
        if elt_thr is None or len(elt_thr) == 0:
            print(f"  ⚠️  No impact data for {thr_m}m ({col_thr}) — skipped")
            continue

        ws_evt_thr = elt_thr.values.sum(axis=1).astype(float)

        # Compute per-event RP from THIS threshold's own EVT2 fit — fully consistent
        rp_evt_thr = compute_rp_from_evt2_spliced(ws_evt_thr, fit_thr)

        # Build full unit matrix for simulation
        elt_prov_thr = elt_thr.values @ prov_mat
        impacts_thr  = np.hstack([elt_thr.values, elt_prov_thr, ws_evt_thr[:, None]])

        # RP-bin weights using this threshold's own lam_total and RP values
        w_thr = rp_bin_weights_from_evt2_rp(
            rp_event=rp_evt_thr, lam_total=lam_thr,
            rp_bins=RP_BINS_FOR_WEIGHTS, min_per_bin=5,
        )

        # YLT simulation
        _, ann_sum_thr, ann_max_thr = simulate_ylt_shortform(
            impacts_evt_by_unit=impacts_thr,
            watershed_evt=ws_evt_thr,
            event_ids=elt_thr.index.astype(str).values,
            weights=w_thr,
            lam_total=lam_thr,
            n_years=N_SIM_YEARS,
            seed=RNG_SEED,
        )

        # OEP — last column is WATERSHED::TOTAL
        _, _, oep_thr = curves_from_annual(ann_sum_thr, ann_max_thr, rp_report=RP_REPORT)
        ws_col_idx = oep_thr.shape[0] - 1

        multi_thr_oep[thr_m] = {
            "oep_watershed": [float(v) for v in oep_thr[ws_col_idx]],
            "lam_total": lam_thr,
            "n_events": len(elt_thr),
        }
        is_primary = abs(thr_m - IMPACT_DEPTH_THR_M) < 1e-6
        print(f"  {'★' if is_primary else '·'} {thr_m}m ({col_thr}): "
              f"{len(elt_thr)} events, lam={lam_thr:.3f}/yr, "
              f"median RP={np.nanmedian(rp_evt_thr):.1f}y  {'← primary' if is_primary else ''}")

    # ── Comparison chart ────────────────────────────────────────────────────
    if multi_thr_oep:
        fig, ax = plt.subplots(figsize=(10, 6))
        colours = cm.viridis_r(np.linspace(0.05, 0.95, len(multi_thr_oep)))

        for (thr_m, res), colour in zip(sorted(multi_thr_oep.items()), colours):
            label = f"{int(round(thr_m * 1000))} mm"
            is_primary = abs(thr_m - IMPACT_DEPTH_THR_M) < 1e-6
            ax.plot(RP_REPORT, res["oep_watershed"],
                    label=label,
                    linewidth=2.8 if is_primary else 1.2,
                    linestyle="-" if is_primary else "--",
                    color=colour)

        ax.set_xscale("log")
        ax.set_xlabel("Return Period (years)", fontsize=12)
        ax.set_ylabel("People Affected — watershed OEP", fontsize=12)
        ax.set_title(
            f"Watershed OEP Sensitivity to Flood Depth Threshold\n"
            f"(primary = {int(IMPACT_DEPTH_THR_M * 1000)} mm, bold solid — "
            f"each curve uses its own EVT2 fit + RP)",
            fontsize=12,
        )
        ax.legend(title="Depth threshold", fontsize=9, title_fontsize=9,
                  loc="upper left", ncol=2 if len(multi_thr_oep) > 6 else 1)
        ax.grid(True, which="both", alpha=0.3)
        plt.tight_layout()
        plt.show()

        # ── Export ──────────────────────────────────────────────────────────
        _multi_oep_path = NB5_OUT_DIR / "watershed_oep_multithr.json"
        _payload = {
            "rp_report":      [float(r) for r in RP_REPORT],
            "primary_depth_m": IMPACT_DEPTH_THR_M,
            "thresholds":     {str(thr_m): res for thr_m, res in sorted(multi_thr_oep.items())},
            "source":         "NB5 multi-threshold OEP comparison — each threshold fully independent",
        }
        _multi_oep_path.write_text(json.dumps(_payload, indent=2), encoding="utf-8")
        print(f"\n✅ Multi-threshold watershed OEP exported: {_multi_oep_path}")
    else:
        print("No threshold results computed — chart skipped.")

In [ ]:
# =========================
# NB5 — Population denominators (NB4-mirrored)
# Defines:
#   pop_exposed_by_muni, pop_total_by_muni
#   pop_exposed_by_prov, pop_total_by_prov
#   pop_exposed_watershed, pop_total_watershed
#   den_exposed (unit order: munis | prov | watershed)
# =========================

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.merge import merge
from rasterio import features as rio_features
from shapely.geometry import box as sbox
from pathlib import Path

# -------------------------
# 0) Preconditions
# -------------------------
required = ["RAW_ROOT", "muni_in_basin", "ADM3_ID_COL", "ADM2_NAME_COL", "IMPACT_DEPTH_THR_M"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Missing required variables: {missing}. Run basin/admin setup cells first.")

# Municipality list in stable order (prefer fid_list from ELT cell if available)
if "fid_list" not in globals():
    fid_list = sorted(muni_in_basin[ADM3_ID_COL].astype(int).unique().tolist())
else:
    fid_list = [int(x) for x in fid_list]

# Province mapping (build if missing)
if "fid_to_adm2" not in globals():
    fid_to_adm2 = muni_in_basin.set_index(ADM3_ID_COL)[ADM2_NAME_COL].to_dict()
if "prov_names" not in globals():
    prov_names = sorted(set(fid_to_adm2.values()))

# -------------------------
# 1) Helper functions (copied in spirit from NB4)
# -------------------------
def reproject_worldpop_to_grid(
    pop_src: np.ndarray,
    pop_src_transform,
    pop_src_crs,
    dst_shape: tuple[int, int],
    dst_transform,
    dst_crs,
    pop_src_nodata=None,
) -> np.ndarray:
    """NB4 parity: bilinear + nodata -> nan, then clip negatives."""
    out = np.full(dst_shape, np.nan, dtype="float32")
    reproject(
        source=pop_src.astype("float32"),
        destination=out,
        src_transform=pop_src_transform,
        src_crs=pop_src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        src_nodata=pop_src_nodata,
        dst_nodata=np.nan,
    )
    out = np.where(np.isfinite(out) & (out < 0), 0.0, out)
    return out

def rasterize_admin_ids_to_grid(admin_gdf, dst_shape, dst_transform, id_field):
    """NB4 parity: all_touched=False."""
    rast = rio_features.rasterize(
        ((geom, int(val)) for geom, val in zip(admin_gdf.geometry, admin_gdf[id_field])),
        out_shape=dst_shape,
        transform=dst_transform,
        fill=0,
        dtype="int32",
        all_touched=False
    )
    ids = np.unique(rast)
    ids = ids[ids > 0]
    return rast, ids

# -------------------------
# 2) Locate inputs (prefer cfg if present; else search RAW_ROOT)
# -------------------------
WORLDPOP_TIF = None
if "cfg" in globals() and isinstance(cfg, dict):
    for k in ["worldpop_tif", "worldpop_path", "worldpop"]:
        if k in cfg and cfg[k]:
            p = Path(cfg[k])
            if p.exists():
                WORLDPOP_TIF = p
                break

if WORLDPOP_TIF is None:
    # heuristic: pick a likely PHL population raster
    cands = [p for p in Path(RAW_ROOT).rglob("*.tif") if "pop" in p.name.lower()]
    # bias toward Philippines if present in filename/path
    phl = [p for p in cands if "phl" in str(p).lower() or "phil" in str(p).lower()]
    pick = phl if phl else cands
    if not pick:
        raise FileNotFoundError("WorldPop GeoTIFF not found. Set WORLDPOP_TIF explicitly.")
    WORLDPOP_TIF = sorted(pick, key=lambda p: p.stat().st_mtime)[-1]

# JRC RP500 depth tiles
JRC_RP500_TIFS = None
if "cfg" in globals() and isinstance(cfg, dict):
    # try some plausible keys
    for k in ["jrc_rp500_tifs", "jrc_rp500_files", "jrc_rp500_paths"]:
        if k in cfg and cfg[k]:
            lst = [Path(x) for x in cfg[k]]
            lst = [p for p in lst if p.exists()]
            if lst:
                JRC_RP500_TIFS = lst
                break

if JRC_RP500_TIFS is None:
    # heuristic: any tif with "500" and likely flood/depth naming under raw
    all_tifs = list(Path(RAW_ROOT).rglob("*.tif"))
    rp = []
    for p in all_tifs:
        s = str(p).lower()
        if "500" in s and (("depth" in s) or ("flood" in s) or ("inun" in s) or ("jrc" in s)):
            rp.append(p)
    if not rp:
        # fallback: anything with "500"
        rp = [p for p in all_tifs if "500" in str(p).lower()]
    if not rp:
        raise FileNotFoundError("Could not find JRC RP500 GeoTIFFs. Set JRC_RP500_TIFS explicitly.")
    JRC_RP500_TIFS = rp

print("WORLDPOP_TIF:", WORLDPOP_TIF)
print("JRC_RP500_TIFS:", len(JRC_RP500_TIFS))

# -------------------------
# 3) Build bbox in EPSG:4326 (NB4 uses lon/lat grids)
# -------------------------
muni_ll = muni_in_basin
if muni_ll.crs is not None and str(muni_ll.crs).lower() not in ["epsg:4326", "crs84"]:
    muni_ll = muni_ll.to_crs("EPSG:4326")
minx, miny, maxx, maxy = map(float, muni_ll.total_bounds)
bbox_geom = sbox(minx, miny, maxx, maxy)

# -------------------------
# 4) Mosaic RP500 tiles clipped to bbox + nodata -> nan
# -------------------------
def tile_intersects_bbox(tif_path: Path, bbox) -> bool:
    try:
        with rasterio.open(tif_path) as src:
            b = src.bounds
            return sbox(b.left, b.bottom, b.right, b.top).intersects(bbox)
    except Exception:
        return True

tifs_use = [p for p in JRC_RP500_TIFS if tile_intersects_bbox(p, bbox_geom)]
if not tifs_use:
    tifs_use = JRC_RP500_TIFS

srcs = [rasterio.open(p) for p in tifs_use]
mosaic, rp500_transform = merge(srcs, bounds=(minx, miny, maxx, maxy))
profile = srcs[0].profile.copy()
nodata_rp = srcs[0].nodata
for s in srcs:
    s.close()

rp500 = mosaic[0].astype("float32")
if nodata_rp is not None:
    rp500 = np.where(rp500 == nodata_rp, np.nan, rp500)

haz_crs = profile["crs"]

# -------------------------
# 5) Reproject WorldPop to RP500 grid (NB4 parity)
# -------------------------
with rasterio.open(WORLDPOP_TIF) as pop_ds:
    pop_src = pop_ds.read(1).astype("float32")
    pop_src_transform = pop_ds.transform
    pop_src_crs = pop_ds.crs
    pop_src_nodata = pop_ds.nodata

pop_on_haz = reproject_worldpop_to_grid(
    pop_src=pop_src,
    pop_src_transform=pop_src_transform,
    pop_src_crs=pop_src_crs,
    dst_shape=rp500.shape,
    dst_transform=rp500_transform,
    dst_crs=haz_crs,
    pop_src_nodata=pop_src_nodata,
)

# NB4 aggregation parity: treat nodata as 0 contribution
pop_on_haz = np.nan_to_num(pop_on_haz, nan=0.0).astype("float32")
pop_on_haz = np.clip(pop_on_haz, 0.0, None)

# -------------------------
# 6) Rasterize muni IDs to same grid (NB4 parity)
# -------------------------
muni_haz = muni_ll
if muni_haz.crs is not None and haz_crs is not None and muni_haz.crs != haz_crs:
    muni_haz = muni_haz.to_crs(haz_crs)

admin_id_raster, admin_ids_present = rasterize_admin_ids_to_grid(
    admin_gdf=muni_haz,
    dst_shape=rp500.shape,
    dst_transform=rp500_transform,
    id_field=ADM3_ID_COL,
)

# -------------------------
# 7) Compute exposed (depth >= 0.2m) and totals
# -------------------------
wet = np.isfinite(rp500) & (rp500 >= float(IMPACT_DEPTH_THR_M))

ids_flat = admin_id_raster.ravel()
valid = ids_flat > 0
ids_valid = ids_flat[valid].astype(np.int32)

exp_flat = np.where(wet, pop_on_haz, 0.0).ravel()[valid]
tot_flat = pop_on_haz.ravel()[valid]

max_id = int(admin_id_raster.max())
sum_exp = np.bincount(ids_valid, weights=exp_flat, minlength=max_id + 1)
sum_tot = np.bincount(ids_valid, weights=tot_flat, minlength=max_id + 1)

pop_exposed_by_muni = {int(i): float(sum_exp[int(i)]) for i in admin_ids_present}
pop_total_by_muni   = {int(i): float(sum_tot[int(i)]) for i in admin_ids_present}

pop_exposed_by_prov = {p: 0.0 for p in prov_names}
pop_total_by_prov   = {p: 0.0 for p in prov_names}

for mid in admin_ids_present:
    mid = int(mid)
    p = fid_to_adm2.get(mid, None)
    if p is None:
        continue
    pop_exposed_by_prov[p] += pop_exposed_by_muni.get(mid, 0.0)
    pop_total_by_prov[p]   += pop_total_by_muni.get(mid, 0.0)

pop_exposed_watershed = float(sum(pop_exposed_by_muni.values()))
pop_total_watershed   = float(sum(pop_total_by_muni.values()))

# Build den_exposed in unit order: munis | prov | watershed
den_exposed = []
for fid in fid_list:
    den_exposed.append(pop_exposed_by_muni.get(int(fid), 0.0))
for p in prov_names:
    den_exposed.append(pop_exposed_by_prov.get(p, 0.0))
den_exposed.append(pop_exposed_watershed)
den_exposed = np.array(den_exposed, dtype=float)

# -------------------------
# 8) Sanity checks (catch the 1e12 bug immediately)
# -------------------------
print(f"PopTotal_watershed:   {pop_total_watershed:,.0f}")
print(f"PopExposed500(depth>={IMPACT_DEPTH_THR_M}m): {pop_exposed_watershed:,.0f}")

if pop_total_watershed > 5e8:
    raise RuntimeError(
        "Population totals are unrealistically large (>> 500 million). "
        "This usually means wrong raster picked or resampling/misaligned CRS. "
        "Check WORLDPOP_TIF and JRC_RP500_TIFS selection printed above."
    )

In [ ]:
import numpy as np
import pandas as pd

# ---- Event process explainability (watershed-level) ----
p_no_event_year = float(np.exp(-lam_total))  # P(N=0) under Poisson
mean_events_year = float(ylt_df["n_events"].mean())
pct_years_any_event = float((ylt_df["n_events"] > 0).mean())
pct_years_any_watershed_impact = float((ylt_df["annual_sum_watershed"] > 0).mean())

# ---- Unit-level coverage (explains zeros per municipality) ----
evt_hits_unit = (impacts_evt_by_unit > 0)
n_footprints_total = int(impacts_evt_by_unit.shape[0])
n_footprints_hit = evt_hits_unit.sum(axis=0).astype(int)
share_footprints_hit = (n_footprints_hit / max(n_footprints_total, 1)).astype(float)

years_impacted = (annual_sum > 0)
share_years_impacted = years_impacted.mean(axis=0).astype(float)

# ---- Population denominators ----
# Ensure den_exposed exists (computed in PopExposed cell) and aligns with unit_names.
if "den_exposed" not in globals() or len(np.atleast_1d(den_exposed)) != len(unit_names):
    if "pop_exposed_by_muni" not in globals():
        raise RuntimeError("pop_exposed_by_muni not found. Run the PopExposed cell before this one.")
    den_exposed = []
    for fid in fid_list:
        den_exposed.append(pop_exposed_by_muni.get(int(fid), 0.0))
    for p in prov_names:
        den_exposed.append(pop_exposed_by_prov.get(p, 0.0))
    den_exposed.append(pop_exposed_watershed)
    den_exposed = np.array(den_exposed, dtype=float)
else:
    den_exposed = np.array(den_exposed, dtype=float)

den_total = []
for fid in fid_list:
    den_total.append(pop_total_by_muni.get(int(fid), 0.0))
for p in prov_names:
    den_total.append(pop_total_by_prov.get(p, 0.0))
den_total.append(pop_total_watershed)
den_total = np.array(den_total, dtype=float)

# ---- Assemble a single unit_stats table for export (munis/prov/watershed) ----
rp25_idx = RP_REPORT.index(ACTION_RP_CAP)

# A stable unit key that allows filtering by ID as well as display label
unit_admin_id = [int(fid) for fid in fid_list] + [str(p) for p in prov_names] + ["TOTAL"]

unit_stats = pd.DataFrame({
    "unit": unit_names,
    "unit_admin_id": unit_admin_id,
    "PopExposed500": den_exposed,
    "PopTotal_bbox": den_total,
    "AAPA": aapa,
    "OEP_RP25": oep_rl[:, rp25_idx],
    "AEP_RP25": aep_rl[:, rp25_idx],
    "share_years_impacted": share_years_impacted,
    "n_footprints_hit": n_footprints_hit,
    "share_footprints_hit": share_footprints_hit,
})

def split_unit(u: str):
    if "::" in u:
        lvl, name = u.split("::", 1)
        return lvl, name
    return "OTHER", u

unit_stats[["level", "name"]] = unit_stats["unit"].apply(lambda u: pd.Series(split_unit(u)))
unit_stats = unit_stats[["level", "name", "unit", "unit_admin_id",
                         "PopExposed500", "PopTotal_bbox", "AAPA", "OEP_RP25", "AEP_RP25",
                         "share_years_impacted", "n_footprints_hit", "share_footprints_hit"]]

# Convenience filtered views (full + practitioner-UI filtered)
muni_stats = unit_stats[unit_stats["level"] == "ADM3"].copy()
prov_stats = unit_stats[unit_stats["level"] == "ADM2"].copy()
watershed_stats = unit_stats[unit_stats["level"] == "WATERSHED"].copy()

if EXCLUDE_ZERO_IMPACT_UNITS and "keep_muni" in globals() and "keep_prov" in globals():
    muni_stats_ui = muni_stats[muni_stats["unit_admin_id"].apply(lambda x: bool(keep_muni.loc[int(x)]))].copy()
    prov_stats_ui = prov_stats[prov_stats["unit_admin_id"].apply(lambda x: bool(keep_prov.loc[str(x)]))].copy()
else:
    muni_stats_ui = muni_stats.copy()
    prov_stats_ui = prov_stats.copy()

display(watershed_stats)
display(muni_stats_ui.sort_values("AAPA", ascending=False).head(10))

In [ ]:
# ---- QA: OEP check against intended exceedance probability 1/RP (watershed) ----
qa_rows = []
ws_row = unit_names.index("WATERSHED::TOTAL")
for rp in RP_REPORT:
    p_target = 1.0 / rp
    # By definition of quantile: P(annual_max <= RL) ≈ 1 - 1/rp
    rl = float(oep_rl[ws_row, RP_REPORT.index(rp)])
    p_emp = float((annual_max[:, ws_row] > rl).mean())  # empirical exceed prob at the RL
    qa_rows.append({"RP": rp, "target_exceed_prob": p_target, "empirical_exceed_prob_at_RL": p_emp,
                    "abs_error": abs(p_emp - p_target)})

qa_oep = pd.DataFrame(qa_rows)
display(qa_oep)

In [ ]:
# --- Cell 10 — Build risk-matrix lookup + portfolio summary (interactive + portfolio) ---

import numpy as np
import pandas as pd

# Ensure den_exposed exists (computed in PopExposed cell) and aligns with unit_names.
if "den_exposed" not in globals() or len(np.atleast_1d(den_exposed)) != len(unit_names):
    den_exposed = []
    for fid in fid_list:
        den_exposed.append(pop_exposed_by_muni.get(int(fid), 0.0))
    for p in prov_names:
        den_exposed.append(pop_exposed_by_prov.get(p, 0.0))
    den_exposed.append(pop_exposed_watershed)
    den_exposed = np.array(den_exposed, dtype=float)
else:
    den_exposed = np.array(den_exposed, dtype=float)

rp_grid = np.array(RP_REPORT, dtype=float)

# --- Risk lookup table: (unit, sev_pct) -> threshold_people, rp_exceed, likelihood_bin ---
risk_rows = []
for ui, uname in enumerate(unit_names):
    denom = float(den_exposed[ui])
    for pct in SEV_PCT_BINS:
        thr = (pct / 100.0) * denom
        rp_ex = invert_oep_curve_to_rp(thr, rp_grid, oep_rl[ui])
        row_bin = likelihood_bin_from_rp(rp_ex, LIKELIHOOD_RP_BINS)

        if not np.isfinite(rp_ex):
            rp_disp = f">{int(max(RP_REPORT))}"
        else:
            rp_disp = f">{int(max(RP_REPORT))}" if rp_ex > max(RP_REPORT) else float(rp_ex)

        risk_rows.append({
            "key": f"{uname}|{pct}",
            "unit": uname,
            "sev_pct": float(pct),
            "threshold_people": float(thr),
            "rp_exceed_num": float(rp_ex) if np.isfinite(rp_ex) else np.inf,
            "rp_exceed_display": rp_disp,
            "likelihood_bin": int(row_bin),
        })

risk_table = pd.DataFrame(risk_rows)

# --- Curves lookup: (unit, RP) -> OEP/AEP for plotting without macros ---
curve_rows = []
for ui, uname in enumerate(unit_names):
    for j, rp in enumerate(RP_REPORT):
        curve_rows.append({
            "key": f"{uname}|{int(rp)}",
            "unit": uname,
            "RP": int(rp),
            "OEP_people": float(oep_rl[ui][j]),
            "AEP_people": float(aep_rl[ui][j]),
        })
# Add intermediate RP targets needed for the interactive EP-style matrix (e.g., RP=3) via log-RP interpolation.
# This keeps Excel formulas simple (direct lookup by unit|RP) and avoids in-sheet interpolation.
extra_rps = [int(rp) for rp in globals().get("RP_TARGETS_MATRIX", [2,3,5,10,20]) if int(rp) not in RP_REPORT]
if extra_rps:
    rp_low, rp_high = 2, 5
    if (rp_low in RP_REPORT) and (rp_high in RP_REPORT):
        j_low = RP_REPORT.index(rp_low)
        j_high = RP_REPORT.index(rp_high)
        for ui, uname in enumerate(unit_names):
            o_low, o_high = float(oep_rl[ui][j_low]), float(oep_rl[ui][j_high])
            a_low, a_high = float(aep_rl[ui][j_low]), float(aep_rl[ui][j_high])
            for rp in extra_rps:
                t = (np.log(float(rp)) - np.log(float(rp_low))) / (np.log(float(rp_high)) - np.log(float(rp_low)))
                curve_rows.append({
                    "key": f"{uname}|{int(rp)}",
                    "unit": uname,
                    "RP": int(rp),
                    "OEP_people": float(o_low + (o_high - o_low) * t),
                    "AEP_people": float(a_low + (a_high - a_low) * t),
                })
    else:
        print("⚠️ Could not add extra RP targets: RP_REPORT must include 2 and 5 for interpolation.")

curves_lookup = pd.DataFrame(curve_rows)

# --- RP2 threshold table (OEP at RP=TRIGGER_RP) ---
rp_trig_idx = RP_REPORT.index(TRIGGER_RP)
threshold_table = pd.DataFrame({
    "unit": unit_names,
    "threshold_people_RP2_OEP": [float(oep_rl[i][rp_trig_idx]) for i in range(len(unit_names))],
    "AAPA_people": [float(aapa[i]) for i in range(len(unit_names))],
    "PopExposed500_thr": [float(den_exposed[i]) for i in range(len(unit_names))],
})

# --- Practitioner-facing unit lists (remove never-affected units from UI only) ---
n_muni = len(fid_list)
n_prov = len(prov_names)

muni_units_all = list(unit_names[:n_muni])
prov_units_all = list(unit_names[n_muni:n_muni+n_prov])

if EXCLUDE_ZERO_IMPACT_UNITS and "keep_muni" in globals() and "keep_prov" in globals():
    muni_units = [u for u, fid in zip(muni_units_all, fid_list) if bool(keep_muni.loc[int(fid)])]
    prov_units = [u for u, p in zip(prov_units_all, prov_names) if bool(keep_prov.loc[str(p)])]
else:
    muni_units = muni_units_all
    prov_units = prov_units_all

choices_units = muni_units + prov_units + ["WATERSHED::TOTAL"]

# --- Portfolio point definition (municipalities) + 5x5 counts grid ---
REF_PCT_FOR_LIKELIHOOD = 2.0     # "chance exceed 2% exposed"
REF_RP_FOR_CONSEQUENCE = ACTION_RP_CAP
rp_conseq_idx = RP_REPORT.index(REF_RP_FOR_CONSEQUENCE)

muni_lookup = muni_in_basin.set_index(ADM3_ID_COL)[ADM3_NAME_COL].to_dict()

portfolio = []
for i, fid in enumerate(fid_list):
    if EXCLUDE_ZERO_IMPACT_UNITS and "keep_muni" in globals():
        if not bool(keep_muni.loc[int(fid)]):
            continue

    denom = float(den_exposed[i])
    thr = (REF_PCT_FOR_LIKELIHOOD / 100.0) * denom
    rp_ex = invert_oep_curve_to_rp(thr, rp_grid, oep_rl[i])
    row_bin = likelihood_bin_from_rp(rp_ex, LIKELIHOOD_RP_BINS)

    impact_rp = float(oep_rl[i][rp_conseq_idx])
    conseq_pct = 0.0 if denom <= 0 else (100.0 * impact_rp / denom)

    if impact_rp < MIN_IMPACT_PERSONS:
        col_bin = 0  # force Minor — absolute floor to prevent misclassification of tiny populations
    else:
        col_bin = int(np.searchsorted(SEV_PCT_BINS, conseq_pct, side="right") - 1)
        col_bin = max(0, min(col_bin, len(SEV_PCT_BINS) - 1))

    portfolio.append({
        "fid": int(fid),
        "muni": muni_lookup.get(int(fid), "UNKNOWN"),
        "province": fid_to_adm2[int(fid)],
        "PopExposed500": float(denom),
        "rp_exceed_2pct_exposed": float(rp_ex) if np.isfinite(rp_ex) else np.inf,
        f"impact_at_RP{REF_RP_FOR_CONSEQUENCE}": impact_rp,
        f"consequence_pct_at_RP{REF_RP_FOR_CONSEQUENCE}": conseq_pct,
        "likelihood_bin": int(row_bin),
        "consequence_bin": int(col_bin),
    })

portfolio_df = pd.DataFrame(portfolio)

counts = np.zeros((5, 5), dtype=int)
for _, r in portfolio_df.iterrows():
    counts[int(r["likelihood_bin"]), int(r["consequence_bin"])] += 1

portfolio_counts = pd.DataFrame(
    counts,
    index=[
        "RP≤2 — Moderate (≥50%/yr)",
        "RP 2–5 — High (20–50%/yr)",
        "RP 5–10 — Very High (10–20%/yr)",
        "RP 10–20 (~5–10%/yr)",
        "RP>20 (<~5%/yr)",
    ],
    columns=[f"{p}%" for p in SEV_PCT_BINS],
)

# --- Excluded lists for audit sheet (0 impacted across all events) ---
excluded_muni_df = pd.DataFrame({
    "adm3_id": [int(x) for x in excluded_muni_ids],
    "municipality": [admin_id_to_name.get(int(x), "UNKNOWN") for x in excluded_muni_ids],
    "province": [fid_to_adm2.get(int(x), "UNKNOWN") for x in excluded_muni_ids],
    "max_event_impact_people": [0.0 for _ in excluded_muni_ids],
    "PopExposed500": [float(pop_exposed_by_muni.get(int(x), 0.0)) for x in excluded_muni_ids],
    "PopTotal_bbox": [float(pop_total_by_muni.get(int(x), 0.0)) for x in excluded_muni_ids],
})

excluded_prov_df = pd.DataFrame({
    "province": [str(x) for x in excluded_prov_names],
    "max_event_impact_people": [0.0 for _ in excluded_prov_names],
    "PopExposed500": [float(pop_exposed_by_prov.get(str(x), 0.0)) for x in excluded_prov_names],
    "PopTotal_bbox": [float(pop_total_by_prov.get(str(x), 0.0)) for x in excluded_prov_names],
})

display(portfolio_df.head())
display(portfolio_counts)

# ── Export watershed OEP curve for NB6 consumption ────────────────────────────
# NB6 uses this curve to classify named event severity by inverting OEP(RP) → RP(pop).
# Execution order: NB5 must run before NB6.
_ws_idx = unit_names.index("WATERSHED::TOTAL")
_oep_curve = {
    "rp": [float(r) for r in RP_REPORT],
    "oep_people": [float(oep_rl[_ws_idx][j]) for j in range(len(RP_REPORT))],
    "source": "NB5 Risk Profiles — watershed-level OEP curve",
}
_oep_curve_path = NB5_OUT_DIR / "watershed_oep_curve.json"
import json as _json_oep
_oep_curve_path.write_text(_json_oep.dumps(_oep_curve, indent=2), encoding="utf-8")
print(f"✅ Watershed OEP curve exported: {_oep_curve_path}")
print(f"   RP range: {_oep_curve['rp'][0]:.0f} – {_oep_curve['rp'][-1]:.0f}")
print(f"   OEP range: {min(_oep_curve['oep_people']):,.0f} – {max(_oep_curve['oep_people']):,.0f} people")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Export full OEP/AEP curves for all units → consumed by NB07 Trigger Validation
# Output: data/processed/Riskprofiles/oep_curves_all_units.json
# ─────────────────────────────────────────────────────────────────────────────
import json as _json_all_units

_all_curves_path = NB5_OUT_DIR / "oep_curves_all_units.json"

_payload = {
    "run_meta": {
        "basin_id": basin_id,
        "run_tag":  run_tag,
        "generated": str(pd.Timestamp.now()),
        "rp_report": [float(r) for r in RP_REPORT],
    },
    "rp_report": [float(r) for r in RP_REPORT],
    "units": [
        {
            "unit":        unit_names[i],
            "level":       unit_stats.loc[i, "level"],
            "name":        unit_stats.loc[i, "name"],
            "pop_exposed": float(den_exposed[i]),
            "pop_total":   float(den_total[i]),
            "aapa":        float(aapa[i]),
            "oep_rl":      [float(v) for v in oep_rl[i]],
            "aep_rl":      [float(v) for v in aep_rl[i]],
        }
        for i in range(len(unit_names))
    ],
}

_all_curves_path.write_text(_json_all_units.dumps(_payload, indent=2), encoding="utf-8")
print(f"✅ NB07 export: {len(unit_names)} unit OEP/AEP curves → {_all_curves_path}")
n_muni = sum(1 for u in unit_names if u.startswith("ADM3::"))
n_prov = sum(1 for u in unit_names if u.startswith("ADM2::"))
print(f"   Municipalities: {n_muni}  |  Provinces: {n_prov}  |  Watershed: 1")
print(f"   RP range: {RP_REPORT[0]}–{RP_REPORT[-1]}  |  {len(RP_REPORT)} points per curve")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — STAKEHOLDER-READY EXCEL EXPORT
# Redesigned for: LGUs, NGO coordinators, NDRRMC, International donors/reviewers
# Design principles:
#   • Plain-language labels throughout — no jargon for non-technical readers
#   • AAPA is the hero metric — highlighted with colour scale on all tables
#   • Narrative sheet order: BRIEFING → DASHBOARD → data sheets → GLOSSARY
#   • All 10,000-row technical sheets hidden; QA/audit data preserved but out of sight
#   • Alert level (VERY HIGH/HIGH/MODERATE/MONITOR) derived and colour-coded on every table
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
from pathlib import Path
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.chart import LineChart, ScatterChart, Reference
from openpyxl.chart import Series as ChartSeries
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

xlsx_path = NB5_OUT_DIR / f"RiskProfile_{basin_id}_{run_tag}.xlsx"

# ── Colour palette (humanitarian / DRM context) ──────────────────────────────
C_NAVY      = "1F4E79"   # dark navy — main headers
C_BLUE      = "2E75B6"   # medium blue — section headers
C_PALE      = "EBF3FB"   # very light blue — alternating rows / intro blocks
C_WHITE     = "FFFFFF"
C_GREY_BG   = "F2F2F2"
C_RED       = "C00000"   # VERY HIGH alert
C_ORANGE    = "F4772E"   # HIGH alert (distinct orange step)
C_AMBER     = "FFC000"   # MODERATE alert
C_GREEN     = "70AD47"   # Monitor / low consequence
C_GREY_MAT  = "44546A"   # below-threshold in TC matrix

# ── Style helpers ─────────────────────────────────────────────────────────────
def _fill(hex6):
    return PatternFill("solid", start_color=hex6, end_color=hex6)

def _font(bold=False, size=10, color=C_NAVY, italic=False, name="Calibri"):
    return Font(name=name, bold=bold, size=size, color=color, italic=italic)

def _align(h="left", v="center", wrap=False):
    return Alignment(horizontal=h, vertical=v, wrap_text=wrap)

def _border(color="BDD7EE"):
    s = Side(style="thin", color=color)
    return Border(left=s, right=s, top=s, bottom=s)

def _header_cell(cell, text, bg=C_NAVY, size=10, wrap=False):
    cell.value = text
    cell.fill = _fill(bg)
    cell.font = _font(bold=True, size=size, color=C_WHITE)
    cell.alignment = _align("center", "center", wrap=wrap)
    cell.border = _border(C_WHITE)

def _section_title(ws, row, col, text, span=1, bg=C_BLUE):
    cell = ws.cell(row, col, text)
    cell.fill = _fill(bg)
    cell.font = _font(bold=True, size=11, color=C_WHITE)
    cell.alignment = _align("left", "center")
    cell.border = _border(C_WHITE)
    if span > 1:
        ws.merge_cells(start_row=row, start_column=col,
                       end_row=row, end_column=col + span - 1)

def _risk_fill_font(val):
    """Return (fill, font) for an alert level string."""
    v = str(val).upper()
    if v == "VERY HIGH":
        return _fill(C_RED), _font(bold=True, size=10, color=C_WHITE)
    elif v == "HIGH":
        return _fill(C_ORANGE), _font(bold=True, size=10, color=C_WHITE)
    elif v == "MODERATE":
        return _fill(C_AMBER), _font(bold=True, size=10, color=C_WHITE)
    else:
        return _fill(C_GREEN), _font(bold=True, size=10, color=C_WHITE)

def _write_table(ws, df, start_row, start_col, col_widths=None, num_fmts=None, risk_col=None):
    """Write styled DataFrame table. risk_col = 0-based column index for risk-level colouring."""
    cols = list(df.columns)
    # Header
    for j, h in enumerate(cols):
        _header_cell(ws.cell(start_row, start_col + j), h, wrap=True)
    # Data
    for i, row_vals in enumerate(df.itertuples(index=False), start=1):
        bg = C_PALE if i % 2 == 0 else C_WHITE
        r = start_row + i
        for j, val in enumerate(row_vals):
            cell = ws.cell(r, start_col + j, val)
            cell.border = _border()
            cell.alignment = _align("right" if j > 0 else "left")
            cell.font = _font(size=10)
            if risk_col is not None and j == risk_col:
                f, fnt = _risk_fill_font(str(val))
                cell.fill = f
                cell.font = fnt
            else:
                cell.fill = _fill(bg)
            if num_fmts and j < len(num_fmts) and num_fmts[j]:
                cell.number_format = num_fmts[j]
    # Column widths
    if col_widths:
        for j, w in enumerate(col_widths):
            if w:
                ws.column_dimensions[get_column_letter(start_col + j)].width = w
    return start_row + len(df) + 1

def _risk_level(aapa_val, pop_exp_val):
    """Compute alert level from AAPA / PopExposed ratio.
    Thresholds derived from Oxfam Pilipinas / PDRRN Cagayan AA framework.
    Absolute floor: units with AAPA < MIN_AAPA_PERSONS are always MONITOR."""
    if aapa_val < MIN_AAPA_PERSONS:
        return "MONITOR"
    if pop_exp_val > 0 and (aapa_val / pop_exp_val) >= 0.05:
        return "VERY HIGH"
    elif pop_exp_val > 0 and (aapa_val / pop_exp_val) >= 0.02:
        return "HIGH"
    elif pop_exp_val > 0 and (aapa_val / pop_exp_val) >= 0.005:
        return "MODERATE"
    return "MONITOR"

# ── Pre-compute key values ────────────────────────────────────────────────────
ws_idx    = unit_names.index("WATERSHED::TOTAL")
rp25_idx  = RP_REPORT.index(ACTION_RP_CAP)
rp1_idx   = RP_REPORT.index(1)   # kept for data display in tables
rp2_idx   = RP_REPORT.index(2)   # Moderate alert threshold
rp5_idx   = RP_REPORT.index(5)   # High alert threshold
rp10_idx  = RP_REPORT.index(10)  # Very High alert threshold

ws_aapa      = float(aapa[ws_idx])
ws_oep_rp1   = float(oep_rl[ws_idx, rp1_idx])
ws_oep_rp2   = float(oep_rl[ws_idx, rp2_idx])
ws_oep_rp5   = float(oep_rl[ws_idx, rp5_idx])
ws_oep_rp10  = float(oep_rl[ws_idx, rp10_idx])
ws_oep_rp25  = float(oep_rl[ws_idx, rp25_idx])
ws_aep_rp25  = float(aep_rl[ws_idx, rp25_idx])
ws_pop_exp   = float(den_exposed[ws_idx])

# ── Load reference events from event viewer population CSVs ─────────────────────────────
_ref_events_csv = PROCESSED_ROOT / "event_viewer" / basin_id / run_tag / "population" / "affected_population_summary.csv"
ref_df = pd.DataFrame(columns=["event_id", "label", "pop_total", "rp_from_oep", "ep_from_oep"])
if _ref_events_csv.exists():
    _rd = pd.read_csv(_ref_events_csv)[["event_id", "label", "pop_total"]]
    _ws_oep_ref = oep_rl[unit_names.index("WATERSHED::TOTAL")]
    _rd["rp_from_oep"] = _rd["pop_total"].apply(
        lambda pop: invert_oep_curve_to_rp(float(pop), np.array(RP_REPORT, dtype=float), _ws_oep_ref)
    )
    _rd["ep_from_oep"] = _rd["rp_from_oep"].apply(
        lambda r: min(1.0, 1.0 / r) if r > 0 and r < float("inf") else 0.0
    )
    ref_df = _rd.sort_values("pop_total").reset_index(drop=True)
    print(f"✅ Loaded {len(ref_df)} reference events from {_ref_events_csv.name}")
else:
    print(f"⚠ Reference events CSV not found: {_ref_events_csv}")

# Province quick summary for BRIEFING
prov_briefing = []
for p in prov_names:
    if EXCLUDE_ZERO_IMPACT_UNITS and "keep_prov" in globals() and not bool(keep_prov.loc[p]):
        continue
    ui       = unit_names.index(f"ADM2::{p}")
    aapa_p   = float(aapa[ui])
    oep1_p   = float(oep_rl[ui, rp1_idx])
    pop_p    = float(den_exposed[ui])
    pct_yr_p = float(share_years_impacted[ui])
    prov_briefing.append({
        "province": p, "aapa": aapa_p, "oep_rp1": oep1_p,  # RP1 (Moderate alert)
        "pop_exp": pop_p, "pct_yrs": pct_yr_p,
        "risk_level": _risk_level(aapa_p, pop_p),
    })
prov_briefing.sort(key=lambda x: x["aapa"], reverse=True)

# Prepare municipality display table
muni_display = muni_stats_ui.sort_values("AAPA", ascending=False).copy()
muni_display["Province"] = muni_display["unit_admin_id"].apply(
    lambda x: fid_to_adm2.get(int(x), "") if str(x).isdigit() else "")
muni_display["Alert Level"] = muni_display.apply(
    lambda r: _risk_level(r["AAPA"], r["PopExposed500"]), axis=1)

# Prepare province display table
prov_display = prov_stats_ui.sort_values("AAPA", ascending=False).copy()
prov_display["Alert Level"] = prov_display.apply(
    lambda r: _risk_level(r["AAPA"], r["PopExposed500"]), axis=1)

# ── Create workbook ───────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

# ═══════════════════════════════════════════════════════════════════════════════
# HIDDEN LOOKUP SHEETS  (identical to original — no changes to formulas/data)
# ═══════════════════════════════════════════════════════════════════════════════
if "choices_units" not in globals():
    muni_units    = [u for u in unit_names if u.startswith("ADM3::")]
    prov_units    = [u for u in unit_names if u.startswith("ADM2::")]
    choices_units = muni_units + prov_units + ["WATERSHED::TOTAL"]

units_range = f"=_lists!$A$1:$A${len(choices_units)}"

ws_lists = wb.create_sheet("_lists")
ws_lists.sheet_state = "hidden"
for i, u in enumerate(choices_units, 1):
    ws_lists.cell(i, 1, u)
for i, u in enumerate(muni_units, 1):
    ws_lists.cell(i, 2, u)

ws_risk = wb.create_sheet("_risk_lookup")
ws_risk.sheet_state = "hidden"
for ri, row in enumerate(dataframe_to_rows(risk_table, index=False, header=True), 1):
    for ci, val in enumerate(row, 1):
        ws_risk.cell(ri, ci, val)

ws_curve = wb.create_sheet("_curves_lookup")
ws_curve.sheet_state = "hidden"
for ri, row in enumerate(dataframe_to_rows(curves_lookup, index=False, header=True), 1):
    for ci, val in enumerate(row, 1):
        ws_curve.cell(ri, ci, val)

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 1 — BRIEFING
# One-page decision brief: context + hero numbers + province traffic lights + navigation
# ═══════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("BRIEFING")
ws.sheet_view.showGridLines = False
for col, w in [("A",3),("B",28),("C",22),("D",22),("E",22),("F",22),("G",22),("H",3)]:
    ws.column_dimensions[col].width = w

# Title banner
ws.merge_cells("B2:G2")
c = ws["B2"]
c.value = f"FLOOD RISK PROFILE — {basin_id.upper().replace('_',' ')}"
c.font = Font(name="Calibri", bold=True, size=20, color=C_WHITE)
c.fill = _fill(C_NAVY)
c.alignment = _align("center", "center")
ws.row_dimensions[2].height = 40

ws.merge_cells("B3:G3")
c = ws["B3"]
c.value = (f"Cagayan River Basin  ·  Run: {run_tag}  ·  "
           f"Flood depth threshold: {IMPACT_DEPTH_THR_M} m  ·  Simulation: {N_SIM_YEARS:,} years")
c.font = Font(name="Calibri", size=10, italic=True, color=C_WHITE)
c.fill = _fill(C_BLUE)
c.alignment = _align("center", "center")
ws.row_dimensions[3].height = 18

# Purpose statement
ws.row_dimensions[4].height = 6
ws.merge_cells("B5:G5")
c = ws["B5"]
c.value = ("This profile estimates how many people could be affected by floods in the Cagayan basin "
           "under different flood scenarios. It supports anticipatory action decisions before a flood occurs.")
c.font = Font(name="Calibri", size=10, italic=True, color="404040")
c.fill = _fill(C_PALE)
c.alignment = _align("left", "center", wrap=True)
ws.row_dimensions[5].height = 36

# Hero metric header
ws.row_dimensions[6].height = 6
ws.merge_cells("B7:G7")
c = ws["B7"]
c.value = "KEY NUMBERS  —  Entire Cagayan Basin"
c.font = Font(name="Calibri", bold=True, size=12, color=C_WHITE)
c.fill = _fill(C_NAVY)
c.alignment = _align("left", "center")
ws.row_dimensions[7].height = 22

# 3 hero metric boxes
hero_specs = [
    ("B8:C11", f"{ws_aapa:,.0f}",    "Avg. People Affected\nper Year",          C_BLUE),
    ("D8:E11", f"{ws_oep_rp2:,.0f}", f"Moderate Flood Year\n(RP2 — 50%/yr annual chance)", C_AMBER),
    ("F8:G11", f"{ws_oep_rp25:,.0f}",f"1-in-{ACTION_RP_CAP}-Year\nWorst Case",  C_RED),
]
for span, val, lbl, bg in hero_specs:
    ws.merge_cells(span)
    r_start = int(span.split(":")[0][1:])
    c_start = span.split(":")[0][0]
    cell = ws[f"{c_start}{r_start}"]
    cell.value = val
    cell.font = Font(name="Calibri", bold=True, size=26, color=C_WHITE)
    cell.fill = _fill(bg)
    cell.alignment = _align("center", "center")
for h in [8, 9, 10, 11]:
    ws.row_dimensions[h].height = 20 if h in (9, 10) else 24

# Hero labels row
hero_labels = [("B12:C12","Avg. affected / year"), ("D12:E12","Moderate alert (RP2 threshold)"), ("F12:G12",f"1-in-{ACTION_RP_CAP}-yr flood")]
for merge_r, lbl in hero_labels:
    ws.merge_cells(merge_r)
    col_start = merge_r.split(":")[0][0]
    cell = ws[f"{col_start}12"]
    cell.value = lbl
    cell.font = Font(name="Calibri", bold=True, size=9, color="595959")
    cell.fill = _fill(C_GREY_BG)
    cell.alignment = _align("center", "center")
ws.row_dimensions[12].height = 16

# Province summary table
ws.row_dimensions[13].height = 8
ws.merge_cells("B14:G14")
c = ws["B14"]
c.value = "FLOOD EXPOSURE BY PROVINCE"
c.font = Font(name="Calibri", bold=True, size=12, color=C_WHITE)
c.fill = _fill(C_NAVY)
c.alignment = _align("left", "center")
ws.row_dimensions[14].height = 22

prov_hdr = ["Province","Avg. Affected/Year","Moderate Year (RP1)","Population in Flood Zone","% Years Affected","Alert Level"]
for j, h in enumerate(prov_hdr):
    _header_cell(ws.cell(15, j+2), h, bg=C_BLUE, size=9, wrap=True)
ws.row_dimensions[15].height = 30

for i, ps in enumerate(prov_briefing):
    r  = 16 + i
    bg = C_PALE if i % 2 == 0 else C_WHITE
    vals = [ps["province"], ps["aapa"], ps["oep_rp1"], ps["pop_exp"], ps["pct_yrs"], ps["risk_level"]]
    fmts = [None, "#,##0", "#,##0", "#,##0", "0.0%", None]
    for j, (v, fmt) in enumerate(zip(vals, fmts)):
        cell = ws.cell(r, j+2, v)
        cell.border = _border()
        cell.font = _font(size=10)
        cell.alignment = _align("center" if j > 0 else "left")
        if j == 5:
            f, fnt = _risk_fill_font(str(v))
            cell.fill = f; cell.font = fnt
        else:
            cell.fill = _fill(bg)
        if fmt:
            cell.number_format = fmt
    ws.row_dimensions[r].height = 18

# Navigation guide
r_nav = 16 + len(prov_briefing) + 2
ws.row_dimensions[r_nav - 1].height = 8
ws.merge_cells(f"B{r_nav}:G{r_nav}")
c = ws[f"B{r_nav}"]
c.value = "HOW TO NAVIGATE THIS FILE"
c.font = Font(name="Calibri", bold=True, size=12, color=C_WHITE)
c.fill = _fill(C_NAVY)
c.alignment = _align("left", "center")
ws.row_dimensions[r_nav].height = 22

nav_items = [
    ("DASHBOARD",       "Model parameters and basin-wide summary metrics at a glance."),
    ("MUNICIPALITIES",  "All affected municipalities ranked by average annual impact. Key table for operational resource allocation."),
    ("PROVINCES",       "Province-level summary. For coordination, reporting, and cross-jurisdiction decisions."),
    ("WATERSHED",       "Full basin totals with exceedance curves — the flood frequency/severity relationship."),
    ("RISK MATRIX",     "Interactive: select any municipality or province to see its exceedance curve and risk position."),
    ("GLOSSARY",        "Plain-language definitions of every technical term used in this file."),
]
for i, (tab, desc) in enumerate(nav_items):
    r  = r_nav + 1 + i
    bg = C_PALE if i % 2 == 0 else C_WHITE
    c_tab = ws.cell(r, 2, f"→  {tab}")
    c_tab.font = Font(name="Calibri", bold=True, size=10, color=C_NAVY)
    c_tab.fill = _fill(bg); c_tab.border = _border()
    c_tab.alignment = _align("left", "center")
    ws.merge_cells(f"C{r}:G{r}")
    c_desc = ws.cell(r, 3, desc)
    c_desc.font = _font(size=10, color="404040")
    c_desc.fill = _fill(bg); c_desc.border = _border()
    c_desc.alignment = _align("left", "center", wrap=True)
    ws.row_dimensions[r].height = 22

# Footer
r_foot = r_nav + len(nav_items) + 3
ws.merge_cells(f"B{r_foot}:G{r_foot}")
c = ws[f"B{r_foot}"]
c.value = (f"Data: GloFAS reforecast library + WorldPop + JRC Flood Hazard Maps  |  "
           f"Alert levels: Moderate RP1 (≥100%/yr) · High RP2 (≥50%/yr) · Very High RP5 (≥20%/yr)  |  "
           f"Actionability cap: RP{ACTION_RP_CAP}  |  Source: Oxfam Pilipinas / PDRRN Start READY DRF")
c.font = Font(name="Calibri", size=8, italic=True, color="808080")
c.alignment = _align("center")

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 2 — DASHBOARD
# Clean parameter summary + basin metrics + Top N municipalities with AAPA colour scale
# ═══════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("DASHBOARD")
ws.sheet_view.showGridLines = False
for col, w in [("A",3),("B",32),("C",20),("D",20),("E",20),("F",20),("G",3)]:
    ws.column_dimensions[col].width = w

ws.merge_cells("B2:F2")
c = ws["B2"]
c.value = f"FLOOD RISK DASHBOARD — {basin_id.upper().replace('_',' ')}"
c.font = Font(name="Calibri", bold=True, size=16, color=C_WHITE)
c.fill = _fill(C_NAVY); c.alignment = _align("center","center")
ws.row_dimensions[2].height = 34

def _param_rows(ws, params, start_r, start_c=2, span_label=3, span_value=2):
    for i, (label, value) in enumerate(params):
        r  = start_r + i
        bg = C_PALE if i % 2 == 0 else C_WHITE
        ws.merge_cells(start_row=r, start_column=start_c, end_row=r, end_column=start_c+span_label-1)
        lc = ws.cell(r, start_c, label)
        lc.font = _font(size=10, color="404040"); lc.fill = _fill(bg)
        lc.border = _border(); lc.alignment = _align("left")
        ws.merge_cells(start_row=r, start_column=start_c+span_label, end_row=r, end_column=start_c+span_label+span_value-1)
        vc = ws.cell(r, start_c+span_label, value)
        vc.font = _font(bold=True, size=10, color=C_NAVY); vc.fill = _fill(bg)
        vc.border = _border(); vc.alignment = _align("right")
        ws.row_dimensions[r].height = 18
    return start_r + len(params)

_section_title(ws, 4, 2, "MODEL PARAMETERS", span=5)
ws.row_dimensions[4].height = 20
model_params = [
    ("Basin",                                   basin_id.replace("_"," ")),
    ("Run tag",                                 run_tag),
    ("Flood depth threshold",                   f"{IMPACT_DEPTH_THR_M} m"),
    ("Simulation length",                       f"{N_SIM_YEARS:,} years"),
    ("Average flood events per year (λ)",       f"{lam_total:.3f}"),
    ("Probability of a flood-free year",        f"{p_no_event_year:.1%}"),
    ("Simulated avg. events per year",          f"{mean_events_year:.2f}"),
    ("Years with at least one flood event",     f"{pct_years_any_event:.1%}"),
    ("Years with any watershed impact",         f"{pct_years_any_watershed_impact:.1%}"),
]
r_after = _param_rows(ws, model_params, 5)

ws.row_dimensions[r_after].height = 8
_section_title(ws, r_after+1, 2, "BASIN-WIDE IMPACT SUMMARY (ENTIRE CAGAYAN WATERSHED)", span=5)
ws.row_dimensions[r_after+1].height = 20
basin_params = [
    ("Population in flood-prone zone (RP500 wet mask)",      f"{ws_pop_exp:,.0f}"),
    ("Average people affected per year — AAPA",              f"{ws_aapa:,.0f}"),
    (f"Moderate flood year — OEP at RP{TRIGGER_RP} (Moderate alert threshold)", f"{ws_oep_rp2:,.0f}"),
    (f"Worst single event — 1-in-{ACTION_RP_CAP}-yr flood (OEP RP{ACTION_RP_CAP})", f"{ws_oep_rp25:,.0f}"),
    (f"Annual total — 1-in-{ACTION_RP_CAP}-yr flood (AEP RP{ACTION_RP_CAP})", f"{ws_aep_rp25:,.0f}"),
    ("Number of historical flood templates used",            f"{n_footprints_total:,}"),
]
r_after2 = _param_rows(ws, basin_params, r_after+2)

# Top municipalities table with AAPA colour scale
ws.row_dimensions[r_after2].height = 8
_section_title(ws, r_after2+1, 2, f"TOP {TOP_N_MUNI_BAR} MUNICIPALITIES — RANKED BY AVERAGE ANNUAL IMPACT (AAPA)", span=5)
ws.row_dimensions[r_after2+1].height = 20

top_munis = muni_stats_ui.sort_values("AAPA", ascending=False).head(TOP_N_MUNI_BAR).copy()
top_munis["Province"] = top_munis["unit_admin_id"].apply(
    lambda x: fid_to_adm2.get(int(x), "") if str(x).isdigit() else "")
top_df = top_munis[["name","Province","PopExposed500","AAPA","OEP_RP25","share_years_impacted"]].copy()
top_df.columns = ["Municipality","Province","Population in Flood Zone",
                   "Avg. Affected/Year (AAPA)","Worst-Case 25yr Flood","% Years Affected"]
top_start = r_after2 + 2
_write_table(ws, top_df, top_start, 2,
             col_widths=[24,16,20,22,20,16],
             num_fmts=[None,None,"#,##0","#,##0","#,##0","0.0%"])

# Colour scale on AAPA column (col E = index 4, start_col=2 → col 5 = E)
aapa_col = get_column_letter(5)
ws.conditional_formatting.add(
    f"{aapa_col}{top_start+1}:{aapa_col}{top_start+TOP_N_MUNI_BAR}",
    ColorScaleRule(start_type="min",  start_color="FFFFFF",
                   mid_type="percentile", mid_value=50, mid_color="FFC000",
                   end_type="max",    end_color="C00000"))
ws.freeze_panes = "B5"

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 3 — MUNICIPALITIES
# Full table sorted by AAPA, colour scale on AAPA, Alert Level column, freeze + notes
# ═══════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("MUNICIPALITIES")
ws.sheet_view.showGridLines = False
for col, w in [("A",3),("B",26),("C",16),("D",18),("E",22),("F",20),("G",20),("H",16),("I",18),("J",12),("K",3)]:
    ws.column_dimensions[col].width = w

ws.merge_cells("B2:J2")
c = ws["B2"]
c.value = "MUNICIPALITIES — Flood Risk Profile"
c.font = Font(name="Calibri", bold=True, size=14, color=C_WHITE)
c.fill = _fill(C_NAVY); c.alignment = _align("left","center")
ws.row_dimensions[2].height = 28

ws.merge_cells("B3:J3")
c = ws["B3"]
c.value = ("Sorted by average annual impact (AAPA ▼). Colour scale on 'Avg. Affected/Year': white (lowest) → orange → red (highest). "
           "Only municipalities with at least one recorded flood impact are shown.")
c.font = Font(name="Calibri", size=9, italic=True, color="404040")
c.fill = _fill(C_PALE); c.alignment = _align("left","center",wrap=True)
ws.row_dimensions[3].height = 30
ws.row_dimensions[4].height = 8

muni_tbl = muni_display[["name","Province","PopExposed500","AAPA","OEP_RP25","AEP_RP25",
                          "share_years_impacted","share_footprints_hit","Alert Level"]].copy()
muni_tbl.columns = ["Municipality","Province","Population in Flood Zone",
                     "Avg. Affected/Year (AAPA) ▼","Worst Single Event\n(RP25 — 4%/yr)",
                     "Annual Total\n(RP25 — 4%/yr)","% of Years\nAffected",
                     "% of Flood Patterns\nCovering Area","Alert Level"]
_write_table(ws, muni_tbl, 5, 2,
             col_widths=None,
             num_fmts=[None,None,"#,##0","#,##0","#,##0","#,##0","0.0%","0.0%",None],
             risk_col=8)

# Colour scale on AAPA (col E = 5th col, start_col=2 → column 5 = E)
n_muni = len(muni_tbl)
ws.conditional_formatting.add(
    f"E6:E{5+n_muni}",
    ColorScaleRule(start_type="min",  start_color="FFFFFF",
                   mid_type="percentile", mid_value=50, mid_color="FFC000",
                   end_type="max",    end_color="C00000"))
ws.freeze_panes = "B6"

r_note = 6 + n_muni + 1
ws.merge_cells(f"B{r_note}:J{r_note}")
c = ws[f"B{r_note}"]
c.value = ("Note: 'Population in Flood Zone' = WorldPop within JRC RP500 wet mask (depth > 0 m). "
           "'Avg. Affected/Year' = simulated annual average over 10,000 years. "
           "Alert Level: VERY HIGH = AAPA > 5% of flood-zone pop.; HIGH = 2–5%; MODERATE = 0.5–2%; MONITOR < 0.5%. "
           "Thresholds: Oxfam Pilipinas / PDRRN Cagayan AA Framework (Start READY DRF Project).")
c.font = Font(name="Calibri", size=8, italic=True, color="808080")
c.alignment = _align("left", wrap=True)
ws.row_dimensions[r_note].height = 36

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 4 — PROVINCES
# ═══════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("PROVINCES")
ws.sheet_view.showGridLines = False
for col, w in [("A",3),("B",24),("C",20),("D",22),("E",20),("F",20),("G",18),("H",18),("I",12),("J",3)]:
    ws.column_dimensions[col].width = w

ws.merge_cells("B2:I2")
c = ws["B2"]
c.value = "PROVINCES — Flood Risk Profile"
c.font = Font(name="Calibri", bold=True, size=14, color=C_WHITE)
c.fill = _fill(C_NAVY); c.alignment = _align("left","center")
ws.row_dimensions[2].height = 28

ws.merge_cells("B3:I3")
c = ws["B3"]
c.value = ("Province-level summary sorted by average annual impact. "
           "For cross-jurisdiction coordination and reporting.")
c.font = Font(name="Calibri", size=9, italic=True, color="404040")
c.fill = _fill(C_PALE); c.alignment = _align("left","center")
ws.row_dimensions[3].height = 20
ws.row_dimensions[4].height = 8

prov_tbl = prov_display[["name","PopExposed500","AAPA","OEP_RP25","AEP_RP25",
                          "share_years_impacted","share_footprints_hit","Alert Level"]].copy()
prov_tbl.columns = ["Province","Population in Flood Zone",
                     "Avg. Affected/Year (AAPA) ▼","Worst Single Event\n(RP25 — 4%/yr)",
                     "Annual Total\n(RP25 — 4%/yr)","% of Years\nAffected",
                     "% of Flood Patterns\nCovering Area","Alert Level"]
_write_table(ws, prov_tbl, 5, 2,
             num_fmts=[None,"#,##0","#,##0","#,##0","#,##0","0.0%","0.0%",None],
             risk_col=7)

n_prov = len(prov_tbl)
ws.conditional_formatting.add(
    f"D6:D{5+n_prov}",
    ColorScaleRule(start_type="min",  start_color="FFFFFF",
                   mid_type="percentile", mid_value=50, mid_color="FFC000",
                   end_type="max",    end_color="C00000"))
ws.freeze_panes = "B6"

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 5 — WATERSHED
# AAPA + full OEP/AEP table + chart
# ═══════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("WATERSHED")
ws.sheet_view.showGridLines = False
for col, w in [("A",3),("B",38),("C",22),("D",22),("E",3)]:
    ws.column_dimensions[col].width = w

ws.merge_cells("B2:D2")
c = ws["B2"]
c.value = "WATERSHED — Full Basin Risk Profile"
c.font = Font(name="Calibri", bold=True, size=14, color=C_WHITE)
c.fill = _fill(C_NAVY); c.alignment = _align("left","center")
ws.row_dimensions[2].height = 28

ws.merge_cells("B3:D3")
c = ws["B3"]
c.value = ("The table below shows, for each return period, how many people across the whole basin "
           "could be affected. OEP = worst single event in the year. AEP = total for all events in the year.")
c.font = Font(name="Calibri", size=9, italic=True, color="404040")
c.fill = _fill(C_PALE); c.alignment = _align("left","center",wrap=True)
ws.row_dimensions[3].height = 36
ws.row_dimensions[4].height = 8

_section_title(ws, 5, 2, "AVERAGE ANNUAL IMPACT (AAPA)", span=3)
ws.row_dimensions[5].height = 20

ws.merge_cells("B6:C6")
c = ws["B6"]
c.value = "Average people affected per year (all floods combined)"
c.font = _font(size=10, color="404040"); c.fill = _fill(C_PALE); c.border = _border(); c.alignment = _align("left")
c = ws["D6"]
c.value = ws_aapa; c.number_format = "#,##0"
c.font = _font(bold=True, size=14, color=C_NAVY); c.fill = _fill(C_PALE); c.border = _border(); c.alignment = _align("right")
ws.row_dimensions[6].height = 24

ws.row_dimensions[7].height = 8
_section_title(ws, 8, 2, "EXCEEDANCE CURVES — Return Period vs People Affected", span=3)
ws.row_dimensions[8].height = 20

curve_hdrs = ["Return Period\n(years)","OEP — Worst Single Event\n(people affected)","AEP — Full Year Total\n(people affected)"]
for j, h in enumerate(curve_hdrs):
    _header_cell(ws.cell(9, j+2), h, wrap=True)
ws.row_dimensions[9].height = 36

wat_oep_vals = oep_rl[ws_idx]
wat_aep_vals = aep_rl[ws_idx]
for i, (rp, ov, av) in enumerate(zip(RP_REPORT, wat_oep_vals, wat_aep_vals)):
    r  = 10 + i
    bg = C_PALE if i % 2 == 0 else C_WHITE
    for j, (v, fmt) in enumerate([(int(rp),"0"),(float(ov),"#,##0"),(float(av),"#,##0")]):
        cell = ws.cell(r, j+2, v)
        cell.fill = _fill(bg); cell.border = _border()
        cell.font = _font(size=10)
        cell.number_format = fmt
        cell.alignment = _align("center" if j==0 else "right")
    ws.row_dimensions[r].height = 18

# Chart
chart = LineChart()
chart.title = "Watershed — Exceedance Curves (OEP & AEP)"
chart.y_axis.title = "People affected"
chart.x_axis.title = "Return period (years)"
chart.style = 10
data_ref = Reference(ws, min_col=3, max_col=4, min_row=9, max_row=9+len(RP_REPORT))
cats_ref = Reference(ws, min_col=2, min_row=10, max_row=9+len(RP_REPORT))
chart.add_data(data_ref, titles_from_data=True)
chart.set_categories(cats_ref)
chart.width = 22; chart.height = 13

# Alert threshold annotations — flat reference lines as additional data columns
# Each threshold is a constant value repeated across all RP rows, sharing the category axis.
_at_thresholds = [("Moderate (RP2)", ws_oep_rp2, C_AMBER),
                  ("High (RP5)",     ws_oep_rp5, C_ORANGE),
                  ("Very High (RP10)", ws_oep_rp10, C_RED)]
for _k, (_at_lbl, _at_val, _at_hex) in enumerate(_at_thresholds):
    _col = 5 + _k   # cols E, F, G
    ws.cell(9, _col, _at_lbl)                           # header row
    ws.cell(9, _col).font = _font(bold=True, size=8, color="808080")
    for _ri in range(len(RP_REPORT)):
        ws.cell(10 + _ri, _col, _at_val)
    ws.column_dimensions[get_column_letter(_col)].width = 2   # narrow — data only, not for reading

# Extend chart data reference to include threshold columns (E, F, G)
# Write reference events as thin horizontal dashed lines (non-overwhelming)
_ref_extra_cols = 0
if not ref_df.empty:
    for _k2, (_ev_idx2, _ev2) in enumerate(ref_df.sort_values("pop_total").iterrows()):
        _col2 = 5 + len(_at_thresholds) + _k2
        ws.cell(9, _col2, str(_ev2["label"])[:18])
        ws.cell(9, _col2).font = _font(bold=False, size=7, color="909090")
        for _ri2 in range(len(RP_REPORT)):
            ws.cell(10 + _ri2, _col2, float(_ev2["pop_total"]))
        ws.column_dimensions[get_column_letter(_col2)].width = 2
    _ref_extra_cols = len(ref_df)
_n_extra = len(_at_thresholds) + _ref_extra_cols
data_ref = Reference(ws, min_col=3, max_col=4 + _n_extra, min_row=9, max_row=9+len(RP_REPORT))
chart.series = []   # clear and re-add with extended range
chart.add_data(data_ref, titles_from_data=True)
chart.set_categories(cats_ref)

# Style the threshold series as dashed lines with matching alert colours
from openpyxl.chart.series import DataPoint
from openpyxl.drawing.line import LineProperties, LineEndProperties
for _k, (_at_lbl, _at_val, _at_hex) in enumerate(_at_thresholds):
    _s = chart.series[2 + _k]   # first 2 are OEP + AEP
    _s.graphicalProperties.line.dashStyle = "dash"
    _s.graphicalProperties.line.solidFill = _at_hex
    _s.graphicalProperties.line.width = 12700  # 1pt in EMU
# Style reference event lines as thin grey dashed
for _k2e in range(_ref_extra_cols):
    _se = chart.series[2 + len(_at_thresholds) + _k2e]
    _se.graphicalProperties.line.dashStyle = "dash"
    _se.graphicalProperties.line.solidFill = "B0B0B0"
    _se.graphicalProperties.line.width = 6350   # 0.5pt

ws.add_chart(chart, "F5")
ws.freeze_panes = "B10"

# Reference events companion table (plain table below the curve data)
if not ref_df.empty:
    _r_ref_ws = 10 + len(RP_REPORT) + 3
    _section_title(ws, _r_ref_ws, 2, "REFERENCE HISTORICAL EVENTS", span=3)
    ws.row_dimensions[_r_ref_ws].height = 20
    for _j_ref, _h_ref in enumerate(["Event", "People Affected", "RP from Model (years)"]):
        _header_cell(ws.cell(_r_ref_ws+1, _j_ref+2), _h_ref)
    ws.row_dimensions[_r_ref_ws+1].height = 22
    for _ri_ref, _rev in enumerate(ref_df.sort_values("pop_total", ascending=False).itertuples(index=False)):
        rr = _r_ref_ws + 2 + _ri_ref
        bg = C_PALE if _ri_ref % 2 == 0 else C_WHITE
        ws.cell(rr, 2, str(_rev.label)).font = _font(size=10)
        ws.cell(rr, 2).fill = _fill(bg); ws.cell(rr, 2).border = _border(); ws.cell(rr, 2).alignment = _align("left")
        ws.cell(rr, 3, float(_rev.pop_total)).number_format = "#,##0"
        ws.cell(rr, 3).font = _font(size=10); ws.cell(rr, 3).fill = _fill(bg); ws.cell(rr, 3).border = _border(); ws.cell(rr, 3).alignment = _align("right")
        _rp_disp = f">{max(RP_REPORT)}" if not (isinstance(_rev.rp_from_oep, float) and _rev.rp_from_oep < float("inf")) else f"{_rev.rp_from_oep:.1f}"
        ws.cell(rr, 4, _rp_disp).font = _font(size=10)
        ws.cell(rr, 4).fill = _fill(bg); ws.cell(rr, 4).border = _border(); ws.cell(rr, 4).alignment = _align("right")
        ws.row_dimensions[rr].height = 18
    print(f"  ✅ Reference events table added to WATERSHED tab ({len(ref_df)} events)")

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 6 — RISK MATRIX
# Dropdown → EP curves (OEP + AEP) + TC-style 5×5 portfolio grid with plain axis labels
# ═══════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("RISK MATRIX")
ws.sheet_view.showGridLines = False
for col, w in [("A",3),("B",20),("C",16),("D",16),("E",16),("F",16),("G",16),("H",16),("I",3)]:
    ws.column_dimensions[col].width = w

ws.merge_cells("B2:H2")
c = ws["B2"]
c.value = "INTERACTIVE RISK MATRIX — Select a Municipality or Province"
c.font = Font(name="Calibri", bold=True, size=14, color=C_WHITE)
c.fill = _fill(C_NAVY); c.alignment = _align("left","center")
ws.row_dimensions[2].height = 28

ws.merge_cells("B3:H3")
c = ws["B3"]
c.value = ("Use the dropdown to select any area. The exceedance curve below will update. "
           "OEP = worst single flood event in a year. AEP = combined total for the year.")
c.font = Font(name="Calibri", size=9, italic=True, color="404040")
c.fill = _fill(C_PALE); c.alignment = _align("left","center",wrap=True)
ws.row_dimensions[3].height = 28
ws.row_dimensions[4].height = 8

ws["B5"].value = "Select area:"
ws["B5"].font = _font(bold=True, size=11)
ws["B5"].alignment = _align("right","center")
ws.merge_cells("C5:F5")
ws["C5"].value = choices_units[0] if choices_units else "WATERSHED::TOTAL"
ws["C5"].font = _font(bold=True, size=11, color=C_NAVY)
ws["C5"].border = _border(C_NAVY); ws["C5"].alignment = _align("center")
ws.row_dimensions[5].height = 24

dv = DataValidation(type="list", formula1=units_range, allow_blank=False)
ws.add_data_validation(dv); dv.add(ws["C5"])

# EP table (RP targets driving the charts)
ws.row_dimensions[6].height = 8
_section_title(ws, 7, 2, "EXCEEDANCE CURVE — Probability vs People Affected (for selected area)", span=7)
ws.row_dimensions[7].height = 20

ep_hdrs = ["Alert Level","Return Period\n(years)","Annual\nProbability","Worst Single Event\n(OEP)","Annual Total\n(AEP)"]
for j, h in enumerate(ep_hdrs):
    _header_cell(ws.cell(8, j+2), h, wrap=True)
ws.row_dimensions[8].height = 36

RP_MAT = [int(x) for x in globals().get("RP_TARGETS_MATRIX", [2, 5, 10])]
# Alert level colors aligned with LIKELIHOOD_RP_BINS=[2,5,10,20]
_alert_colors = {2: C_AMBER, 5: C_ORANGE, 10: C_RED}
_alert_labels = {2: "Moderate", 5: "High", 10: "Very High"}
for i, rp in enumerate(RP_MAT):
    rr = 9 + i
    bg = C_PALE if i % 2 == 0 else C_WHITE
    alert_color = _alert_colors.get(rp)
    # Alert label cell (col B)
    alert_lbl = ws.cell(rr, 2, _alert_labels.get(rp, f"RP{rp}"))
    alert_lbl.fill = _fill(alert_color) if alert_color else _fill(bg)
    alert_lbl.font = _font(bold=True, size=9, color=C_WHITE if alert_color else C_NAVY)
    alert_lbl.border = _border()
    alert_lbl.alignment = _align("center", "center")
    ws.row_dimensions[rr].height = 20
    for j, (col_offset, formula) in enumerate([
        (0, None),           # RP value in col C (shift right by 1)
        (1, f"=1-1/C{rr}"),
        (2, f'=INDEX(_curves_lookup!$D:$D,MATCH($C$5&"|"&C{rr},_curves_lookup!$A:$A,0))'),
        (3, f'=INDEX(_curves_lookup!$E:$E,MATCH($C$5&"|"&C{rr},_curves_lookup!$A:$A,0))'),
    ]):
        cell = ws.cell(rr, 3+col_offset)   # start at col C (3) to leave B for label
        if j == 0:
            cell.value = int(rp)
        elif formula:
            cell.value = formula
        cell.fill = _fill(bg); cell.border = _border()
        cell.font = _font(size=10)
        cell.alignment = _align("center" if j < 2 else "right")
        cell.number_format = ["0","0.0%","#,##0","#,##0"][j]

# Full RP curve table
r_full = 9 + len(RP_MAT) + 2
_section_title(ws, r_full, 2, "FULL CURVE — All Return Periods", span=7)
ws.row_dimensions[r_full].height = 20
full_hdrs = ["Return Period (years)","Annual Probability","Worst Single Event (OEP)","Annual Total (AEP)"]
for j, h in enumerate(full_hdrs):
    _header_cell(ws.cell(r_full+1, j+2), h)
ws.row_dimensions[r_full+1].height = 22

for i, rp in enumerate(RP_REPORT):
    rr = r_full + 2 + i
    bg = C_PALE if i % 2 == 0 else C_WHITE
    ws.cell(rr,2,int(rp)).number_format = "0"
    for j, (col_off, formula) in enumerate([
        (0, None),
        (1, f"=1-1/B{rr}"),
        (2, f'=INDEX(_curves_lookup!$D:$D,MATCH($C$5&"|"&B{rr},_curves_lookup!$A:$A,0))'),
        (3, f'=INDEX(_curves_lookup!$E:$E,MATCH($C$5&"|"&B{rr},_curves_lookup!$A:$A,0))'),
    ]):
        cell = ws.cell(rr, 2+col_off)
        if formula: cell.value = formula
        cell.fill = _fill(bg); cell.border = _border()
        cell.font = _font(size=10)
        cell.alignment = _align("center" if j < 2 else "right")
        cell.number_format = ["0","0.0%","#,##0","#,##0"][j]
    ws.row_dimensions[rr].height = 18

# EP scatter chart (OEP + AEP + named events from NB6)
# Layout: B=AlertLabel, C=RP, D=AnnualProb, E=OEP, F=AEP
ep_chart = ScatterChart()
ep_chart.title = "Exceedance Probability Curve (selected area)"
ep_chart.y_axis.title = "Annual Probability (= 1 - 1/RP)"
ep_chart.x_axis.title = "People Affected"
ep_chart.y_axis.scaling.min = 0.0; ep_chart.y_axis.scaling.max = 1.0
ep_chart.style = 10
# Use full curve section (all 11 RP points) for a complete EP chart
# Full curve layout: B=RP, C=AnnualProb, D=OEP, E=AEP (data rows r_full+2 onward)
x_oep = Reference(ws, min_col=4, min_row=r_full+2, max_row=r_full+1+len(RP_REPORT))
x_aep = Reference(ws, min_col=5, min_row=r_full+2, max_row=r_full+1+len(RP_REPORT))
y_p   = Reference(ws, min_col=3, min_row=r_full+2, max_row=r_full+1+len(RP_REPORT))
s_oep = ChartSeries(y_p, x_oep, title="OEP — Worst Single Event")
s_aep = ChartSeries(y_p, x_aep, title="AEP — Annual Total")
s_oep.marker.symbol = "circle"; s_aep.marker.symbol = "triangle"
ep_chart.series.extend([s_oep, s_aep])

# Reference events — vertical dashed lines on the EP scatter chart
# Each event: 2 points with same pop_total (x) but y=0 and y=1.0 → vertical line
if not ref_df.empty:
    for _k, (_ev_idx, _ev) in enumerate(ref_df.sort_values("pop_total").iterrows()):
        _col_x = 10 + _k * 2   # cols J, L, N, P … (x = pop_total)
        _col_y = 11 + _k * 2   # cols K, M, O, Q … (y = 0 or 1)
        ws.cell(7, _col_x, float(_ev["pop_total"]))   # top point (EP=1.0)
        ws.cell(7, _col_y, 1.0)
        ws.cell(8, _col_x, float(_ev["pop_total"]))   # bottom point (EP=0.0)
        ws.cell(8, _col_y, 0.0)
        ws.column_dimensions[get_column_letter(_col_x)].hidden = True
        ws.column_dimensions[get_column_letter(_col_y)].hidden = True
        x_ev = Reference(ws, min_col=_col_x, min_row=7, max_row=8)
        y_ev = Reference(ws, min_col=_col_y, min_row=7, max_row=8)
        s_ev = ChartSeries(y_ev, x_ev, title=str(_ev["label"])[:25])
        s_ev.graphicalProperties.line.solidFill = "808080"
        s_ev.graphicalProperties.line.dashStyle = "dash"
        s_ev.graphicalProperties.line.width = 9525   # 0.75pt
        s_ev.marker.symbol = "none"
        ep_chart.series.append(s_ev)
    print(f"  ✅ {len(ref_df)} reference event vertical dashes added to RISK MATRIX EP chart")
else:
    print("  ⚠ No reference events — ref_df is empty (check CSV path)")

ep_chart.width = 22; ep_chart.height = 13
ws.add_chart(ep_chart, "F7")

# TC-style 5x5 portfolio matrix — alert vocabulary + updated likelihood labels
f_grey   = _fill(C_GREY_MAT); f_green  = _fill(C_GREEN)
f_amber  = _fill(C_AMBER);    f_orange = _fill(C_ORANGE); f_red = _fill(C_RED)
tc_grid = [
    [f_amber,  f_orange, f_orange, f_red,    f_red],
    [f_amber,  f_amber,  f_orange, f_orange, f_red],
    [f_grey,   f_grey,   f_amber,  f_orange, f_red],
    [f_grey,   f_grey,   f_grey,   f_amber,  f_orange],
    [f_grey,   f_grey,   f_grey,   f_green,  f_amber],
]
consequence_labels = ["Minor\n(<0.5%)", "Low\n(0.5–1%)", "Moderate\n(1–2%)", "High\n(2–5%)", "Severe\n(>5%)"]
# Likelihood labels aligned with LIKELIHOOD_RP_BINS=[2,5,10,20]
likelihood_labels  = [
    "Very Likely\n≤RP2 (≥50%/yr)",
    "Likely\nRP2–5 (20–50%)",
    "Possible\nRP5–10 (10–20%)",
    "Unlikely\nRP10–20 (5–10%)",
    "Rare\n>RP20 (<5%/yr)",
]

r_mat = r_full + 2 + len(RP_REPORT) + 3
_section_title(ws, r_mat, 2, "PORTFOLIO MATRIX — All Municipalities: Likelihood vs Consequence", span=7)
ws.row_dimensions[r_mat].height = 20

# Axis labels (no spanning merges — avoids MergedCell write errors)
ws.cell(r_mat+1, 2, "Likelihood ↓ / Consequence →").font = _font(bold=True, size=9, color=C_NAVY)
ws.cell(r_mat+1, 2).fill = _fill(C_GREY_BG)
ws.cell(r_mat+1, 2).border = _border()
ws.cell(r_mat+1, 2).alignment = _align("center","center",wrap=True)
ws.row_dimensions[r_mat+1].height = 24

# Consequence column headers
for j, cl in enumerate(consequence_labels):
    _header_cell(ws.cell(r_mat+2, j+3), cl, bg=C_BLUE, size=8, wrap=True)
ws.row_dimensions[r_mat+2].height = 36

# Likelihood row labels + grid (each label in its own cell — no spanning merge)
for i, ll in enumerate(likelihood_labels):
    r = r_mat+3+i
    _header_cell(ws.cell(r, 2), ll, bg=C_BLUE, size=8, wrap=True)
    ws.row_dimensions[r].height = 28
    for j in range(5):
        cell = ws.cell(r, j+3)
        cell.fill = tc_grid[i][j]
        cell.border = _border(C_WHITE)
        cell.value = ""   # cells show colour only; counts deferred to later iteration
        cell.font = Font(name="Calibri", bold=True, size=14, color=C_WHITE)
        cell.alignment = _align("center","center")

# Matrix legend
r_leg = r_mat + 9
ws.cell(r_leg, 2, "Legend:").font = _font(bold=True, size=9)
for k, (lf, lt) in enumerate([
    (f_green,  "Monitor (low consequence / low likelihood)"),
    (f_amber,  "Heightened Readiness — Moderate alert"),
    (f_orange, "Pre-emptive Action — High alert"),
    (f_red,    "Pre-emptive Action — Very High alert"),
    (f_grey,   "Below Actionability Threshold"),
]):
    ws.cell(r_leg+1+k, 2).fill = lf
    ws.cell(r_leg+1+k, 2).border = _border()
    c = ws.cell(r_leg+1+k, 3, lt)
    c.font = _font(size=9, color="404040")
    ws.merge_cells(start_row=r_leg+1+k, start_column=3, end_row=r_leg+1+k, end_column=7)

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 7 — GLOSSARY
# ═══════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("GLOSSARY")
ws.sheet_view.showGridLines = False
ws.column_dimensions["A"].width = 3
ws.column_dimensions["B"].width = 34
ws.column_dimensions["C"].width = 72
ws.column_dimensions["D"].width = 3

ws.merge_cells("B2:C2")
c = ws["B2"]
c.value = "GLOSSARY — Plain-Language Definitions"
c.font = Font(name="Calibri", bold=True, size=14, color=C_WHITE)
c.fill = _fill(C_NAVY); c.alignment = _align("left","center")
ws.row_dimensions[2].height = 28

ws.merge_cells("B3:C3")
c = ws["B3"]
c.value = "Every technical term used in this workbook, explained without jargon."
c.font = Font(name="Calibri", size=9, italic=True, color="404040")
c.fill = _fill(C_PALE); c.alignment = _align("left","center")
ws.row_dimensions[3].height = 18
ws.row_dimensions[4].height = 8

_header_cell(ws.cell(5,2), "Term", size=10)
_header_cell(ws.cell(5,3), "Plain-Language Explanation", size=10)
ws.row_dimensions[5].height = 22

glossary = [
    ("Return Period (RP)",
     "How often a flood of a given size occurs on average. RP10 = roughly once every 10 years, "
     "which means a 10% chance of occurring in any given year. Smaller RP = more frequent."),
    ("Annual Probability",
     "The chance a flood of at least this size happens in any year. Calculated as 1 ÷ Return Period. "
     "RP1 = 100% chance per year (Moderate alert); RP2 = 50% (High); RP5 = 20% (Very High); RP25 = 4% (Actionability cap)."),
    ("OEP — Worst Single Event",
     "Occurrence Exceedance Probability. The number of people affected by the single worst flood event "
     "in a year. This is the most conservative impact estimate for a single event."),
    ("AEP — Annual Total",
     "Aggregate Exceedance Probability. The total number of people affected by all floods combined "
     "in a year. Useful for planning cumulative response capacity."),
    ("AAPA — Avg. Annual People Affected",
     "The long-run average number of people affected by floods per year, averaged across all 10,000 "
     "simulated years (including flood-free years). The headline impact metric in this workbook."),
    ("Population in Flood Zone (PopExposed500)",
     "People living in areas flooded by a 1-in-500-year event (JRC flood hazard map, depth > 0 m). "
     "This is the 'at-risk' population used as the denominator for all percentage calculations."),
    (f"RP{TRIGGER_RP} — Moderate Alert Threshold",
     f"Flood impact at RP{TRIGGER_RP} has a {1/TRIGGER_RP:.0%} annual probability (once per year on average). "
     f"This is the lowest operational alert level in this workbook. Events at or above this threshold "
     f"indicate conditions consistent with the Moderate scenario in the Cagayan AA Framework."),
    (f"RP{ACTION_RP_CAP} Actionability Cap",
     f"The maximum return period considered actionable for pre-emptive decisions. Floods rarer than "
     f"1-in-{ACTION_RP_CAP} years are shown for context but are outside the normal operational planning window."),
    ("YLT — Year Loss Table",
     f"A synthetic catalogue of {N_SIM_YEARS:,} simulated years of flood impacts, generated by sampling "
     "from historical event patterns according to the statistical flood frequency model (EVT2)."),
    ("Footprint / Event Template",
     "A historical reforecast flood event used as a spatial impact pattern. The simulation draws from "
     "these templates to create realistic multi-municipality impact scenarios with realistic spatial correlation."),
    ("Anticipatory Action (AA)",
     "Taking preparedness or pre-positioning actions before a disaster strikes, based on forecast information "
     "and agreed risk thresholds — rather than only responding after the flood has occurred."),
    ("Alert Levels (MODERATE / HIGH / VERY HIGH)",
     "Derived from AAPA ÷ population in flood zone: VERY HIGH = >5%; HIGH = 2–5%; MODERATE = 0.5–2%; MONITOR < 0.5%. "
     "The three named alert levels (MODERATE/HIGH/VERY HIGH) correspond to RP1/RP2/RP5 respectively and are "
     "aligned with the Oxfam Pilipinas / PDRRN Flood Anticipatory Action Framework for the Cagayan River Basin "
     "(Start READY DRF Project, 2025). "
     "Important caveat: these thresholds use the simulated annual average impact (AAPA) as a proxy. "
     "The AA Framework defines scenario levels by event-peak population impact at specific return periods "
     "(e.g. 42% at RP2), which is a different metric. AAPA will always be lower than peak impact at the same RP. "
     "A municipality classified HIGH here may correspond to VERY HIGH under the event-peak definition. "
     "Interpret alert levels as planning guidance, not as literal AA Framework scenario matches."),
    ("% Years Affected",
     "The share of the 10,000 simulated years in which at least one person was affected in that area. "
     "Indicates how frequently the area experiences any flood impact at all."),
    ("% of Flood Patterns Covering Area",
     "The proportion of historical flood event templates that produced any impact in this area. "
     "High values = affected by many different flood scenarios. Low values = only certain types of floods reach this area."),
]

for i, (term, defn) in enumerate(glossary):
    r  = 6 + i
    bg = C_PALE if i % 2 == 0 else C_WHITE
    tc = ws.cell(r, 2, term)
    tc.fill = _fill(bg); tc.border = _border()
    tc.font = _font(bold=True, size=10, color=C_NAVY)
    tc.alignment = _align("left","top",wrap=True)
    dc = ws.cell(r, 3, defn)
    dc.fill = _fill(bg); dc.border = _border()
    dc.font = _font(size=10, color="404040")
    dc.alignment = _align("left","top",wrap=True)
    ws.row_dimensions[r].height = 42

ws.freeze_panes = "B6"

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 8 — TECHNICAL NOTES  (replaces README)
# ═══════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("TECHNICAL NOTES")
ws.sheet_view.showGridLines = False
ws.column_dimensions["A"].width = 3
ws.column_dimensions["B"].width = 28
ws.column_dimensions["C"].width = 80

ws.merge_cells("B2:C2")
c = ws["B2"]
c.value = "NB5 Risk Profile — Technical Notes"
c.font = Font(name="Calibri", bold=True, size=14, color=C_WHITE)
c.fill = _fill(C_NAVY); c.alignment = _align("left","center")
ws.row_dimensions[2].height = 28
ws.row_dimensions[3].height = 8

tech = [
    ("Basin & Run",         f"Basin: {basin_id}  |  Run tag: {run_tag}  |  Depth thr: {IMPACT_DEPTH_THR_M} m  |  Sim years: {N_SIM_YEARS}  |  RNG seed: {RNG_SEED}"),
    ("Population denom.",   "PopExposed500(unit) = WorldPop summed over JRC RP500 wet pixels (depth > 0) within unit & model AOI."),
    ("EVT2 freq. model",    f"λ_total = {lam_total:.4f} events/yr (Poisson). EVT2 body/tail via RP-bin mass weights. Fit from NB4."),
    ("Reforecast library",  "GloFAS historical reforecast events used as spatial templates. Weighted by RP-bin mass from EVT2."),
    ("AEP vs OEP",          "AEP = annual aggregate (sum of events). OEP = annual occurrence (max single event). Both from empirical quantiles of YLT."),
    ("Monotonicity",        "Curves enforced monotone non-decreasing (cumulative max) post-simulation."),
    ("Lookup sheets",       "_lists, _risk_lookup, _curves_lookup are hidden. Risk Matrix dropdown and EP formulas use these."),
    ("Hidden audit sheets", "YLT_AUDIT, FOOTPRINTS, QA_CHECK, EXCLUDED_UNITS: hidden. Unhide via Format > Sheet > Unhide."),
    ("External references", "JRC/CEMS GloFAS FloodHazard maps (RP 10–500yr). WorldPop. HydroBASINS L6."),
]

for i, (heading, content) in enumerate(tech):
    r_h = 4 + i * 3
    c_h = ws.cell(r_h, 2, heading)
    c_h.font = _font(bold=True, size=10, color=C_NAVY); c_h.fill = _fill(C_GREY_BG)
    c_h.border = _border(); c_h.alignment = _align("left")
    ws.merge_cells(f"B{r_h}:C{r_h}")
    ws.row_dimensions[r_h].height = 18
    r_c = r_h + 1
    c_c = ws.cell(r_c, 2, content)
    c_c.font = _font(size=10, color="404040"); c_c.fill = _fill(C_WHITE)
    c_c.border = _border(); c_c.alignment = _align("left",wrap=True)
    ws.merge_cells(f"B{r_c}:C{r_c}")
    ws.row_dimensions[r_c].height = 30
    ws.row_dimensions[r_h+2].height = 4

# ═══════════════════════════════════════════════════════════════════════════════
# HIDDEN TECHNICAL / AUDIT SHEETS
# ═══════════════════════════════════════════════════════════════════════════════
def _add_df_plain(ws_t, df):
    for ri, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for ci, val in enumerate(row, 1):
            ws_t.cell(ri, ci, val)

if EXPORT_YLT_NONZERO:
    ws_ylt = wb.create_sheet("YLT_AUDIT")
    ws_ylt.sheet_state = "hidden"
    _add_df_plain(ws_ylt, ylt_df[ylt_df["n_events"] > 0].copy())

ws_cat = wb.create_sheet("FOOTPRINTS")
ws_cat.sheet_state = "hidden"
_add_df_plain(ws_cat, pd.DataFrame({
    "event_id": event_ids,
    "RP_impact_op_full": rp_evt,
    "PopAffected_op_registry": evt_tbl["PopAffected_op"].values,
    "PopAffected_op_from_impacts": watershed_evt,
    "sampling_weight": w,
}))

if EXPORT_QA_OEP_CHECK:
    ws_qa = wb.create_sheet("QA_CHECK")
    ws_qa.sheet_state = "hidden"
    _add_df_plain(ws_qa, qa_oep)

ws_ex = wb.create_sheet("EXCLUDED_UNITS")
ws_ex.sheet_state = "hidden"
ws_ex["A1"] = "Units excluded from UI (max impacted people = 0 across all events)"
ws_ex["A3"] = "Municipalities"
_add_df_plain(ws_ex, excluded_muni_df)  # starts row 1 — acceptable for hidden audit sheet
r_prov_start = len(excluded_muni_df) + 5
ws_ex[f"A{r_prov_start}"] = "Provinces"
_add_df_plain(ws_ex, excluded_prov_df)

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════════════════════════════════
xlsx_path.parent.mkdir(parents=True, exist_ok=True)
try:
    wb.save(xlsx_path)
except PermissionError:
    from datetime import datetime
    _fallback = xlsx_path.with_name(f"{xlsx_path.stem}_{datetime.now():%Y%m%d_%H%M%S}{xlsx_path.suffix}")
    wb.save(_fallback)
    print(f"⚠ Target file is open/locked. Saved fallback file to: {_fallback}")
    xlsx_path = _fallback

visible_sheets = [s.title for s in wb.worksheets if s.sheet_state != "hidden"]
print(f"✅ Stakeholder-ready workbook saved: {xlsx_path}")
print(f"   Visible sheets ({len(visible_sheets)}): {visible_sheets}")
print(f"   Hidden sheets:  {[s.title for s in wb.worksheets if s.sheet_state == 'hidden']}")
